In [1]:
from google.colab import drive
import json
import numpy as np
import pandas as pd
import torch

!pip -q install -U transformers datasets accelerate evaluate scikit-learn
!pip install -q wandb
!pip install -q pysentimiento
!pip install -q "pyarrow>=14.0.0" --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 21.5 MB/s eta 0:00:00


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
drive.mount('/content/drive')

ruta = "/content/drive/MyDrive/Tesis ORT Benedetti La Laina/df_final_para_modelo.jsonl"

df_limpio = pd.read_json(ruta, lines=True)

Mounted at /content/drive


In [4]:
import os
import wandb

os.environ["WANDB_API_KEY"] = "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"
wandb.login()  # la toma desde el env


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: flalaina (fbenedetti97-universidad-ort-uruguay) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
from sklearn.metrics import classification_report, f1_score, accuracy_score
import numpy as np
import wandb

def one_vs_rest_accuracy(y_true, y_pred, class_id):
    y_true_bin = (np.array(y_true) == class_id).astype(int)
    y_pred_bin = (np.array(y_pred) == class_id).astype(int)
    return accuracy_score(y_true_bin, y_pred_bin)

def build_metrics_dict(y_true, y_pred, label_list, label2id, split, variant):
    """
    split: 'eval' o 'test'
    variant: 'simple' o 'sliding'
    """
    report = classification_report(
        y_true,
        y_pred,
        labels=[label2id[lbl] for lbl in label_list],
        target_names=label_list,
        output_dict=True,
        zero_division=0
    )

    metrics = {
        f"{split}/accuracy_{variant}": accuracy_score(y_true, y_pred),
        f"{split}/f1_macro_{variant}": f1_score(y_true, y_pred, average="macro"),
        f"{split}/f1_weighted_{variant}": f1_score(y_true, y_pred, average="weighted"),
        f"{split}/confusion_matrix_{variant}": wandb.plot.confusion_matrix(
            probs=None,
            y_true=y_true,
            preds=y_pred,
            class_names=label_list
        ),
    }

    for lbl in label_list:
        metrics[f"{split}/class_{lbl}_precision_{variant}"] = float(report[lbl]["precision"])
        metrics[f"{split}/class_{lbl}_recall_{variant}"] = float(report[lbl]["recall"])
        metrics[f"{split}/class_{lbl}_f1_{variant}"] = float(report[lbl]["f1-score"])
        metrics[f"{split}/class_{lbl}_support_{variant}"] = int(report[lbl]["support"])
        metrics[f"{split}/class_{lbl}_accuracy_ovr_{variant}"] = float(
            one_vs_rest_accuracy(y_true, y_pred, label2id[lbl])
        )

    return metrics, report

# ROUBERTA BASELINE SIN ENTRENAR CON INTERVENCIONES PARLAMENTARIAS

In [6]:
# =========================================
# 0) W&B + Config
# =========================================
import os, json, random
import numpy as np
import pandas as pd
import torch
import wandb

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import RobertaModel, RobertaTokenizer

WANDB_PROJECT = "Tesis ORT"
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

if wandb.run is not None:
    wandb.finish()

SEED        = 1899
FOLDER_PATH = "rouberta_sentiment_10_epochs"
MODEL_TAG   = "rouberta_ft_noticias"

USE_SLIDING = False          # True = sliding, False = truncado simple
AGG_METHOD  = "vote"         # solo se usa si USE_SLIDING=True
MAX_LEN_SW  = 128
STRIDE_SW   = 96

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

variant_name = "sliding" if USE_SLIDING else "simple"
mode_name = f"sliding_{AGG_METHOD}" if USE_SLIDING else "simple"
RUN_NAME = f"{MODEL_TAG}_{mode_name}_{SEED}"

run = wandb.init(
    project=WANDB_PROJECT,
    name=RUN_NAME,
    config={
        "seed": int(SEED),
        "model_ckpt": FOLDER_PATH,
        "model_tag": MODEL_TAG,
        "use_sliding": USE_SLIDING,
        "max_len_sw": int(MAX_LEN_SW),
        "stride_sw": int(STRIDE_SW),
        "agg_method": AGG_METHOD if USE_SLIDING else None,
        "run_type": "finetuned_noticias_inference_only",
        "selection_metric": "test/f1_macro_sliding" if USE_SLIDING else "test/f1_macro_simple",
    },
    tags=["baseline", "inference_only", "rouberta", "sliding" if USE_SLIDING else "simple"],
    reinit="finish_previous",
)
print("RUN:", wandb.run.name, "ID:", wandb.run.id)

# =========================================
# 1) Cargar modelo finetuneado con state_dict
# =========================================
with open(f"{FOLDER_PATH}/inference_config.json") as f:
    config = json.load(f)

print("label_map del modelo:", config.get("label_map"))

class RobertaClass(torch.nn.Module):
    def __init__(self):
        super(RobertaClass, self).__init__()
        self.l1 = RobertaModel.from_pretrained(config["model_path"])
        self.pre_classifier = torch.nn.Linear(768, config["hidden_size"])
        self.dropout = torch.nn.Dropout(0.25)
        self.classifier = torch.nn.Linear(config["hidden_size"], config["num_labels"])

    def forward(self, input_ids, attention_mask):
        output_1 = self.l1(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = output_1.last_hidden_state
        pooler = hidden_state[:, 0]
        pooler = self.pre_classifier(pooler)
        pooler = torch.nn.ReLU()(pooler)
        pooler = self.dropout(pooler)
        return self.classifier(pooler)

model = RobertaClass()
state_dict = torch.load(f"{FOLDER_PATH}/model_weights.bin", map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

tokenizer = RobertaTokenizer.from_pretrained(FOLDER_PATH)
print("Modelo cargado desde:", FOLDER_PATH)

# =========================================
# 2) Utils de inferencia
# =========================================
@torch.no_grad()
def predict_proba_windows(text, model, tokenizer, max_len=128, stride=64, device=None):
    """Sliding window real."""
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    if isinstance(input_ids[0], int):
        input_ids = [input_ids]
        attention_mask = [attention_mask]

    max_batch_len = max(len(x) for x in input_ids)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

    def pad(seq, val):
        return seq + [val] * (max_batch_len - len(seq))

    input_ids = torch.tensor(
        [pad(x, pad_id) for x in input_ids],
        dtype=torch.long,
        device=device
    )
    attention_mask = torch.tensor(
        [pad(x, 0) for x in attention_mask],
        dtype=torch.long,
        device=device
    )

    logits = model(input_ids=input_ids, attention_mask=attention_mask)
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    return probs, len(probs)

@torch.no_grad()
def predict_text_with_sliding(text, model, tokenizer, id2label, max_len=128, stride=64, agg="vote"):
    """Predicción con múltiples ventanas."""
    probs_windows, n_win = predict_proba_windows(text, model, tokenizer, max_len, stride)
    votes_counts = None

    if agg == "vote":
        votes = np.argmax(probs_windows, axis=1)
        votes_counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(int)
        max_count = votes_counts.max()
        tied = np.where(votes_counts == max_count)[0]

        if len(tied) == 1:
            pred_id = int(tied[0])
        else:
            pred_id = int(tied[np.argmax(probs_windows.mean(axis=0)[tied])])

        probs_agg = votes_counts.astype(float) / votes_counts.sum()
        return id2label[pred_id], float(probs_agg[pred_id]), probs_agg, n_win, votes_counts

    elif agg == "mean":
        probs_agg = probs_windows.mean(axis=0)
        pred_id = int(np.argmax(probs_agg))
        return id2label[pred_id], float(np.max(probs_agg)), probs_agg, n_win, None

    else:
        raise ValueError("agg debe ser 'vote' o 'mean'")

@torch.no_grad()
def predict_text_truncated(text, model, tokenizer, id2label, max_len=128, device=None):
    """Predicción con truncado simple: una sola pasada por texto."""
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        return_attention_mask=True,
        padding="max_length",
        return_tensors="pt",
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    logits = model(input_ids=input_ids, attention_mask=attention_mask)
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()[0]

    pred_id = int(np.argmax(probs))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs))
    n_win = 1
    votes_counts = None

    return pred_label, conf, probs, n_win, votes_counts

def predict_text(text, model, tokenizer, id2label, use_sliding=True, max_len=128, stride=64, agg="vote"):
    if use_sliding:
        return predict_text_with_sliding(
            text=text,
            model=model,
            tokenizer=tokenizer,
            id2label=id2label,
            max_len=max_len,
            stride=stride,
            agg=agg
        )
    else:
        return predict_text_truncated(
            text=text,
            model=model,
            tokenizer=tokenizer,
            id2label=id2label,
            max_len=max_len
        )

# =========================================
# 3) Datos — alineados al orden del modelo
# label_map modelo: Neg=0, Pos=1, Neu=2
# Nuestros labels: N=0, P=1, Neu=2
# =========================================
df_lab = df_limpio[df_limpio["sentimiento"].notna()].copy()
df_lab["text"] = df_lab["intervencion"].astype(str)
df_lab = df_lab[df_lab["sentimiento"].isin(["N", "Neu", "P"])].copy()

label2id = {
    "N": 0,
    "P": 1,
    "Neu": 2
}
id2label = {
    0: "N",
    1: "P",
    2: "Neu"
}
label_list = ["N", "P", "Neu"]

df_lab["label"] = df_lab["sentimiento"].map(label2id)

print("Mapping usado en inferencia:")
print("label2id =", label2id)
print("id2label =", id2label)
print("Modo de inferencia:", "SLIDING" if USE_SLIDING else "SIMPLE")

train_df, temp_df = train_test_split(
    df_lab,
    test_size=0.20,
    random_state=SEED,
    stratify=df_lab["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

wandb.log({
    "data/n_train": len(train_df),
    "data/n_val": len(val_df),
    "data/n_test": len(test_df)
})
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# =========================================
# 4) Inferencia validation
# =========================================
y_true_val = val_df["label"].to_numpy()
y_pred_val = []
nwin_val = []

for txt in val_df["text"].tolist():
    lab, _, _, n_win, _ = predict_text(
        txt,
        model,
        tokenizer,
        id2label,
        use_sliding=USE_SLIDING,
        max_len=MAX_LEN_SW,
        stride=STRIDE_SW,
        agg=AGG_METHOD
    )
    y_pred_val.append(label2id[lab])
    nwin_val.append(n_win)

y_pred_val = np.array(y_pred_val)

val_metrics, val_report = build_metrics_dict(
    y_true=y_true_val,
    y_pred=y_pred_val,
    label_list=label_list,
    label2id=label2id,
    split="eval",
    variant=variant_name
)

val_metrics.update({
    f"eval/avg_windows_{variant_name}": float(np.mean(nwin_val)),
    f"eval/max_windows_{variant_name}": float(np.max(nwin_val)),
})

wandb.log(val_metrics)

print(
    f"Val accuracy: {val_metrics[f'eval/accuracy_{variant_name}']:.4f} | "
    f"F1-macro: {val_metrics[f'eval/f1_macro_{variant_name}']:.4f} | "
    f"Avg windows: {np.mean(nwin_val):.4f}"
)
print("\nValidation report:")
print(classification_report(y_true_val, y_pred_val, target_names=label_list, zero_division=0))

# =========================================
# 5) Inferencia test
# =========================================
test_meta = test_df.reset_index(drop=True).copy()
y_true = test_meta["label"].to_numpy()

pred_label_list, conf_list, n_windows_all = [], [], []
proba_N_list, proba_P_list, proba_Neu_list = [], [], []
votes_N_list, votes_P_list, votes_Neu_list = [], [], []

for txt in test_meta["text"].tolist():
    lab, conf, probs_agg, n_win, votes_counts = predict_text(
        txt,
        model,
        tokenizer,
        id2label,
        use_sliding=USE_SLIDING,
        max_len=MAX_LEN_SW,
        stride=STRIDE_SW,
        agg=AGG_METHOD
    )

    pred_label_list.append(lab)
    conf_list.append(conf)
    n_windows_all.append(n_win)

    proba_N_list.append(float(probs_agg[label2id["N"]]))
    proba_P_list.append(float(probs_agg[label2id["P"]]))
    proba_Neu_list.append(float(probs_agg[label2id["Neu"]]))

    if votes_counts is None:
        votes_N_list.append(np.nan)
        votes_P_list.append(np.nan)
        votes_Neu_list.append(np.nan)
    else:
        votes_N_list.append(int(votes_counts[label2id["N"]]))
        votes_P_list.append(int(votes_counts[label2id["P"]]))
        votes_Neu_list.append(int(votes_counts[label2id["Neu"]]))

test_meta["true_label"] = test_meta["sentimiento"]
test_meta["pred_label"] = pred_label_list
test_meta["conf_pred"] = conf_list
test_meta["n_windows"] = n_windows_all

test_meta["proba_N"] = proba_N_list
test_meta["proba_P"] = proba_P_list
test_meta["proba_Neu"] = proba_Neu_list
test_meta["proba_vector"] = test_meta[["proba_N", "proba_P", "proba_Neu"]].values.tolist()

test_meta["votes_N"] = votes_N_list
test_meta["votes_P"] = votes_P_list
test_meta["votes_Neu"] = votes_Neu_list

y_pred = test_meta["pred_label"].map(label2id).to_numpy()
df_errors = test_meta[test_meta["true_label"] != test_meta["pred_label"]].copy()

test_metrics, test_report = build_metrics_dict(
    y_true=y_true,
    y_pred=y_pred,
    label_list=label_list,
    label2id=label2id,
    split="test",
    variant=variant_name
)

test_metrics.update({
    f"test/avg_windows_{variant_name}": float(np.mean(n_windows_all)),
    f"test/max_windows_{variant_name}": float(np.max(n_windows_all)),
})

print("\n" + classification_report(y_true, y_pred, target_names=label_list, zero_division=0))
wandb.log(test_metrics)

print("=" * 50)
print(f"  Mode:        {variant_name}")
print(f"  Accuracy:    {test_metrics[f'test/accuracy_{variant_name}']:.4f}")
print(f"  F1-macro:    {test_metrics[f'test/f1_macro_{variant_name}']:.4f}")
print(f"  F1-weighted: {test_metrics[f'test/f1_weighted_{variant_name}']:.4f}")
print(f"  Avg windows: {np.mean(n_windows_all):.4f}")
print(f"  Errores:     {len(df_errors)} / {len(test_meta)}")
print("=" * 50)

for lbl in label_list:
    print(
        f"[TEST] {lbl} | "
        f"Precision: {test_metrics[f'test/class_{lbl}_precision_{variant_name}']:.4f} | "
        f"Recall: {test_metrics[f'test/class_{lbl}_recall_{variant_name}']:.4f} | "
        f"F1: {test_metrics[f'test/class_{lbl}_f1_{variant_name}']:.4f} | "
        f"Support: {int(test_metrics[f'test/class_{lbl}_support_{variant_name}'])} | "
        f"Acc OvR: {test_metrics[f'test/class_{lbl}_accuracy_ovr_{variant_name}']:.4f}"
    )

# =========================================
# 6) Export Excel
# =========================================
cols_excel = [
    "id_intervencion", "file_name", "n_legislatura", "cuerpo", "fecha",
    "locutor", "encabezado", "intervencion", "n_palabras", "n_caracteres",
    "año", "sexo_locutor", "apellido_locutor_norm", "partido_imputado",
    "sentimiento", "text", "true_label", "pred_label", "conf_pred",
    "proba_N", "proba_P", "proba_Neu", "proba_vector",
    "n_windows", "votes_N", "votes_P", "votes_Neu",
]
cols_excel = [c for c in cols_excel if c in test_meta.columns]

out_path = f"analisis_{RUN_NAME}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    test_meta[cols_excel].to_excel(writer, index=False, sheet_name="test_all")
    df_errors[cols_excel].to_excel(writer, index=False, sheet_name="errores")

print("Excel generado:", out_path)
wandb.finish()

RUN: rouberta_ft_noticias_simple_1899 ID: wn8uzsfp
label_map del modelo: {'Neg': 0, 'Pos': 1, 'Neu': 2}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: filevich/robertita-cased
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo cargado desde: rouberta_sentiment_10_epochs
Mapping usado en inferencia:
label2id = {'N': 0, 'P': 1, 'Neu': 2}
id2label = {0: 'N', 1: 'P', 2: 'Neu'}
Modo de inferencia: SIMPLE
Train: 979 | Val: 122 | Test: 123
Val accuracy: 0.7295 | F1-macro: 0.7312 | Avg windows: 1.0000

Validation report:
              precision    recall  f1-score   support

           N       0.78      0.68      0.73        41
           P       0.74      0.78      0.76        36
         Neu       0.69      0.73      0.71        45

    accuracy                           0.73       122
   macro avg       0.73      0.73      0.73       122
weighted avg       0.73      0.73      0.73       122


              precision    recall  f1-score   support

           N       0.63      0.54      0.58        41
           P       0.65      0.72      0.68        36
         Neu       0.58      0.61      0.60        46

    accuracy                           0.62       123
   macro avg       0.62      0.62      0.62    

data/n_test,▁
data/n_train,▁
data/n_val,▁
eval/accuracy_simple,▁
eval/avg_windows_simple,▁
eval/class_N_accuracy_ovr_simple,▁
eval/class_N_f1_simple,▁
eval/class_N_precision_simple,▁
eval/class_N_recall_simple,▁
eval/class_N_support_simple,▁
+33,...


# ROUBERTA SEED 1899

## Truncado

In [ ]:
# =========================================
# 0) W&B + Config (antes de todo)
# =========================================
import os
import random
import numpy as np
import pandas as pd
import torch
import wandb

WANDB_PROJECT = "Tesis ORT"
WANDB_ENTITY  = "fbenedetti97-universidad-ort-uruguay"

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["WANDB_ENTITY"] = WANDB_ENTITY

if wandb.run is not None:
    wandb.finish()

USE_SLIDING = False
AGG_METHOD  = "mean"   # "mean" / "vote"

# =========================================
# 1) Imports + Config
# =========================================
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

SEED       = 1899
MODEL_CKPT = "pln-udelar/rouberta-base-uy22-cased"
MODEL_TAG  = "rouberta"

RUN_NAME   = (
    f"{MODEL_TAG}_base_{SEED}"
    if not USE_SLIDING
    else f"{MODEL_TAG}_slide_{SEED}_{AGG_METHOD}"
)

MAX_LEN_TRAIN = 128
MAX_LEN_SW    = 128
STRIDE_SW     = 96

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================
# 2) Init W&B
# =========================================
run = wandb.init(
    project=WANDB_PROJECT,
    # entity=WANDB_ENTITY,
    name=RUN_NAME,
    config={
        "seed": int(SEED),
        "model_ckpt": MODEL_CKPT,
        "model_tag": MODEL_TAG,
        "run_type": "finetuned_single_run",
        "max_len_train": int(MAX_LEN_TRAIN),
        "use_sliding": USE_SLIDING,
        "max_len_sw": int(MAX_LEN_SW),
        "stride_sw": int(STRIDE_SW),
        "agg_method": AGG_METHOD,
        "learning_rate": float(2e-5),
        "train_batch_size": int(16),
        "eval_batch_size": int(32),
        "epochs": int(4),
        "weight_decay": float(0.01),
        "selection_metric": "test/f1_macro_sliding" if USE_SLIDING else "test/f1_macro_simple",
    },
    tags=["finetune", "single_run", "rouberta", "sliding" if USE_SLIDING else "simple"],
    reinit="finish_previous",
)

print("RUN:", wandb.run.name, "ID:", wandb.run.id, "USE_SLIDING:", USE_SLIDING, "AGG:", AGG_METHOD)

# =========================================
# 3) Utils
# =========================================
def softmax_stable(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

@torch.no_grad()
def predict_proba_windows(
    text: str,
    model,
    tokenizer,
    max_len: int = 128,
    stride: int = 64,
    device=None
):
    model.eval()
    if device is None:
        device = model.device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    if len(input_ids) > 0 and isinstance(input_ids[0], int):
        input_ids = [input_ids]
        attention_mask = [attention_mask]

    max_batch_len = max(len(x) for x in input_ids)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

    def pad(seq, pad_value):
        return seq + [pad_value] * (max_batch_len - len(seq))

    input_ids = torch.tensor(
        [pad(x, pad_id) for x in input_ids],
        dtype=torch.long,
        device=device
    )
    attention_mask = torch.tensor(
        [pad(x, 0) for x in attention_mask],
        dtype=torch.long,
        device=device
    )

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    return probs, len(probs)

@torch.no_grad()
def predict_text_truncated(text, model, tokenizer, id2label, max_len=128, device=None):
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        return_attention_mask=True,
        padding="max_length",
        return_tensors="pt",
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()[0]

    pred_id = int(np.argmax(probs))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs))
    n_win = 1
    votes_counts = None

    return pred_label, conf, probs, n_win, votes_counts

def aggregate_probs(probs_windows: np.ndarray, method: str = "mean"):
    if method == "mean":
        return probs_windows.mean(axis=0)
    if method == "max":
        return probs_windows.max(axis=0)
    if method == "vote":
        votes = np.argmax(probs_windows, axis=1)
        counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(float)
        return counts / counts.sum()
    raise ValueError("method debe ser 'mean', 'max' o 'vote'")

def predict_text_with_sliding(text, model, tokenizer, id2label,
                              max_len=128, stride=64, agg="mean"):
    """
    Retorna:
      pred_label (str),
      conf (float),
      probs_agg (np.array),
      n_win (int),
      votes_counts (np.array int) o None
    """
    probs_windows, n_win = predict_proba_windows(
        text=text,
        model=model,
        tokenizer=tokenizer,
        max_len=max_len,
        stride=stride
    )
    votes_counts = None

    if agg == "vote":
        votes = np.argmax(probs_windows, axis=1)
        votes_counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(int)
        max_count = votes_counts.max()
        tied = np.where(votes_counts == max_count)[0]

        if len(tied) == 1:
            pred_id = int(tied[0])
        else:
            mean_probs = probs_windows.mean(axis=0)
            pred_id = int(tied[np.argmax(mean_probs[tied])])

        probs_agg = votes_counts.astype(float) / votes_counts.sum()
        conf = float(probs_agg[pred_id])
        pred_label = id2label[pred_id]
        return pred_label, conf, probs_agg, n_win, votes_counts

    probs_agg = aggregate_probs(probs_windows, method=agg)
    pred_id = int(np.argmax(probs_agg))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs_agg))
    return pred_label, conf, probs_agg, n_win, votes_counts

# =========================================
# 4) Datos
# =========================================
df_lab = df_limpio[df_limpio["sentimiento"].notna()].copy()
df_lab["text"] = df_lab["intervencion"].astype(str)

valid_labels = ["N", "Neu", "P"]
df_lab = df_lab[df_lab["sentimiento"].isin(valid_labels)].copy()

label_list = ["N", "Neu", "P"]
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
df_lab["label"] = df_lab["sentimiento"].map(label2id)

# Split 80/10/10
train_df, temp_df = train_test_split(
    df_lab,
    test_size=0.20,
    random_state=SEED,
    stratify=df_lab["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

wandb.log({
    "data/n_train": len(train_df),
    "data/n_val": len(val_df),
    "data/n_test": len(test_df),
})

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# =========================================
# 5) Tokenize
# =========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize_train(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN_TRAIN)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

ds = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})
ds_tok = ds.map(tokenize_train, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================================
# 6) Model
# =========================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# =========================================
# 7) Metrics para Trainer
# =========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# =========================================
# 8) Trainer + Train
# =========================================
use_fp16 = torch.cuda.is_available()
OUTPUT_DIR = f"{MODEL_TAG}_finetuned_{wandb.run.id}"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    fp16=use_fp16,
    report_to="wandb",
    run_name=wandb.run.name,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

best_model = trainer.model

# =========================================
# 9) VALIDATION simple — referencia, no selección final
# =========================================
pred_val_simple = trainer.predict(ds_tok["validation"])
logits_val_simple = pred_val_simple.predictions
y_true_val_simple = pred_val_simple.label_ids
y_pred_val_simple = np.argmax(logits_val_simple, axis=-1)

eval_simple_metrics, eval_simple_report = build_metrics_dict(
    y_true=y_true_val_simple,
    y_pred=y_pred_val_simple,
    label_list=label_list,
    label2id=label2id,
    split="eval",
    variant="simple"
)

wandb.log(eval_simple_metrics)

print("\nVALIDATION SIMPLE")
print(f"  Accuracy: {eval_simple_metrics['eval/accuracy_simple']:.4f}")
print(f"  F1-macro: {eval_simple_metrics['eval/f1_macro_simple']:.4f}")
print(classification_report(
    y_true_val_simple,
    y_pred_val_simple,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 10) VALIDATION sliding — referencia, no selección final
# =========================================
if USE_SLIDING:
    y_true_val_sliding = val_df["label"].to_numpy()
    y_pred_val_sw = []
    nwin_val = []

    for txt in val_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_val_sw.append(label2id[lab])
        nwin_val.append(n_win)

    y_pred_val_sw = np.array(y_pred_val_sw)

    eval_sliding_metrics, eval_sliding_report = build_metrics_dict(
        y_true=y_true_val_sliding,
        y_pred=y_pred_val_sw,
        label_list=label_list,
        label2id=label2id,
        split="eval",
        variant="sliding"
    )

    eval_sliding_metrics.update({
        "eval/avg_windows_sliding": float(np.mean(nwin_val)),
        "eval/max_windows_sliding": float(np.max(nwin_val)),
    })

    wandb.log(eval_sliding_metrics)

    print("\nVALIDATION SLIDING")
    print(f"  Accuracy: {eval_sliding_metrics['eval/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {eval_sliding_metrics['eval/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true_val_sliding,
        y_pred_val_sw,
        target_names=label_list,
        zero_division=0
    ))

# =========================================
# 11) TEST simple — resultado final comparable
# =========================================
pred_test_simple = trainer.predict(ds_tok["test"])
logits_test_simple = pred_test_simple.predictions
y_true = pred_test_simple.label_ids
y_pred = np.argmax(logits_test_simple, axis=-1)
probs = softmax_stable(logits_test_simple, axis=1)

true_label_txt = np.array([id2label[i] for i in y_true])
pred_label_txt = np.array([id2label[i] for i in y_pred])

test_meta = test_df.reset_index(drop=True).copy()
test_meta["true_label"] = true_label_txt
test_meta["pred_label"] = pred_label_txt
test_meta["conf_pred"] = probs.max(axis=1)
test_meta["proba_N"] = probs[:, label2id["N"]]
test_meta["proba_Neu"] = probs[:, label2id["Neu"]]
test_meta["proba_P"] = probs[:, label2id["P"]]
test_meta["proba_vector"] = probs.tolist()

test_simple_metrics, test_simple_report = build_metrics_dict(
    y_true=y_true,
    y_pred=y_pred,
    label_list=label_list,
    label2id=label2id,
    split="test",
    variant="simple"
)

wandb.log(test_simple_metrics)

print("\nTEST SIMPLE")
print(f"  Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f}")
print(f"  F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f}")
print(classification_report(
    y_true,
    y_pred,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 12) TEST sliding — resultado final comparable
# =========================================
if USE_SLIDING:
    y_pred_sw = []
    conf_sw = []
    nwin_sw = []
    votes_N_list, votes_Neu_list, votes_P_list = [], [], []

    for txt in test_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_sw.append(label2id[lab])
        conf_sw.append(conf)
        nwin_sw.append(n_win)

        if votes_counts is None:
            votes_N_list.append(np.nan)
            votes_Neu_list.append(np.nan)
            votes_P_list.append(np.nan)
        else:
            votes_N_list.append(int(votes_counts[label2id["N"]]))
            votes_Neu_list.append(int(votes_counts[label2id["Neu"]]))
            votes_P_list.append(int(votes_counts[label2id["P"]]))

    y_pred_sw = np.array(y_pred_sw)

    assert len(test_df) == len(y_true) == len(y_pred_sw), "Mismatch en longitudes de test"

    test_meta["pred_label_sw"] = [id2label[i] for i in y_pred_sw]
    test_meta["conf_sw"] = conf_sw
    test_meta["n_windows"] = nwin_sw
    test_meta["votes_N"] = votes_N_list
    test_meta["votes_Neu"] = votes_Neu_list
    test_meta["votes_P"] = votes_P_list

    test_sliding_metrics, test_sliding_report = build_metrics_dict(
        y_true=y_true,
        y_pred=y_pred_sw,
        label_list=label_list,
        label2id=label2id,
        split="test",
        variant="sliding"
    )

    test_sliding_metrics.update({
        "test/avg_windows_sliding": float(np.mean(nwin_sw)),
        "test/max_windows_sliding": float(np.max(nwin_sw)),
    })

    wandb.log(test_sliding_metrics)

    print("\nTEST SLIDING")
    print(f"  Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true,
        y_pred_sw,
        target_names=label_list,
        zero_division=0
    ))

else:
    test_meta["pred_label_sw"] = np.nan
    test_meta["conf_sw"] = np.nan
    test_meta["n_windows"] = np.nan
    test_meta["votes_N"] = np.nan
    test_meta["votes_Neu"] = np.nan
    test_meta["votes_P"] = np.nan

# =========================================
# 13) Export Excel (test_all + errores simple + errores sliding)
# =========================================
cols_excel = [
    "id_intervencion", "file_name", "n_legislatura", "cuerpo", "fecha",
    "locutor", "encabezado", "intervencion", "n_palabras", "n_caracteres",
    "año", "sexo_locutor", "apellido_locutor_norm", "partido_imputado",
    "sentimiento", "text",
    "true_label",
    "pred_label", "conf_pred", "proba_N", "proba_Neu", "proba_P", "proba_vector",
    "pred_label_sw", "conf_sw", "n_windows", "votes_N", "votes_Neu", "votes_P",
]
cols_excel = [c for c in cols_excel if c in test_meta.columns]

df_errors_simple = test_meta[test_meta["true_label"] != test_meta["pred_label"]].copy()
df_errors_sliding = (
    test_meta[test_meta["true_label"] != test_meta["pred_label_sw"]].copy()
    if USE_SLIDING else pd.DataFrame()
)
df_corrige_sliding = (
    test_meta[
        (test_meta["true_label"] != test_meta["pred_label"]) &
        (test_meta["true_label"] == test_meta["pred_label_sw"])
    ].copy()
    if USE_SLIDING else pd.DataFrame()
)

out_path = f"analisis_{RUN_NAME}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    test_meta[cols_excel].to_excel(writer, index=False, sheet_name="test_all")
    df_errors_simple[cols_excel].to_excel(writer, index=False, sheet_name="errores_simple")
    if USE_SLIDING:
        df_errors_sliding[cols_excel].to_excel(writer, index=False, sheet_name="errores_sliding")
        df_corrige_sliding[cols_excel].to_excel(writer, index=False, sheet_name="corrige_sliding")

print("Excel generado:", out_path)
print(f"  test total:        {len(test_meta)}")
print(f"  errores simple:    {len(df_errors_simple)}")
if USE_SLIDING:
    print(f"  errores sliding:   {len(df_errors_sliding)}")
    print(f"  corrige sliding:   {len(df_corrige_sliding)}")

print("\n=== TEST SIMPLE ===")
print(classification_report(y_true, y_pred, target_names=label_list, zero_division=0))

if USE_SLIDING:
    print("\n=== TEST SLIDING ===")
    print(classification_report(y_true, y_pred_sw, target_names=label_list, zero_division=0))

print("\n" + "=" * 60)
print("RESUMEN FINAL — TEST")
print(f"Simple  | Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f} | "
      f"F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f} | "
      f"F1-weighted: {test_simple_metrics['test/f1_weighted_simple']:.4f}")

if USE_SLIDING:
    print(f"Sliding | Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f} | "
          f"F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f} | "
          f"F1-weighted: {test_sliding_metrics['test/f1_weighted_sliding']:.4f}")
print("=" * 60)

wandb.finish()

RUN: rouberta_base_1899 ID: qthr1x0u USE_SLIDING: False AGG: mean
Train: 979 Val: 122 Test: 123


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/699 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


VALIDATION SIMPLE
  Accuracy: 0.8033
  F1-macro: 0.8000
              precision    recall  f1-score   support

           N       0.74      0.85      0.80        41
         Neu       0.83      0.84      0.84        45
           P       0.86      0.69      0.77        36

    accuracy                           0.80       122
   macro avg       0.81      0.80      0.80       122
weighted avg       0.81      0.80      0.80       122




TEST SIMPLE
  Accuracy: 0.7154
  F1-macro: 0.7036
              precision    recall  f1-score   support

           N       0.66      0.85      0.74        41
         Neu       0.83      0.74      0.78        46
           P       0.66      0.53      0.58        36

    accuracy                           0.72       123
   macro avg       0.71      0.71      0.70       123
weighted avg       0.72      0.72      0.71       123

Excel generado: analisis_rouberta_base_1899.xlsx
  test total:        123
  errores simple:    35

=== TEST SIMPLE ===
              precision    recall  f1-score   support

           N       0.66      0.85      0.74        41
         Neu       0.83      0.74      0.78        46
           P       0.66      0.53      0.58        36

    accuracy                           0.72       123
   macro avg       0.71      0.71      0.70       123
weighted avg       0.72      0.72      0.71       123


RESUMEN FINAL — TEST
Simple  | Accuracy: 0.7154 | F1-macro: 0.7036 

data/n_test,▁
data/n_train,▁
data/n_val,▁
eval/accuracy,▁█▆▆
eval/accuracy_simple,▁
eval/class_N_accuracy_ovr_simple,▁
eval/class_N_f1_simple,▁
eval/class_N_precision_simple,▁
eval/class_N_recall_simple,▁
eval/class_N_support_simple,▁
+48,...


## Slide + Vote

In [ ]:
# =========================================
# 0) W&B + Config (antes de todo)
# =========================================
import os
import random
import numpy as np
import pandas as pd
import torch
import wandb

WANDB_PROJECT = "Tesis ORT"
WANDB_ENTITY  = "fbenedetti97-universidad-ort-uruguay"

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["WANDB_ENTITY"] = WANDB_ENTITY

if wandb.run is not None:
    wandb.finish()

USE_SLIDING = True
AGG_METHOD  = "vote"   # "mean" / "vote"

# =========================================
# 1) Imports + Config
# =========================================
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

SEED       = 1899
MODEL_CKPT = "pln-udelar/rouberta-base-uy22-cased"
MODEL_TAG  = "rouberta"

RUN_NAME   = (
    f"{MODEL_TAG}_base_{SEED}"
    if not USE_SLIDING
    else f"{MODEL_TAG}_slide_{SEED}_{AGG_METHOD}"
)

MAX_LEN_TRAIN = 128
MAX_LEN_SW    = 128
STRIDE_SW     = 96

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================
# 2) Init W&B
# =========================================
run = wandb.init(
    project=WANDB_PROJECT,
    # entity=WANDB_ENTITY,
    name=RUN_NAME,
    config={
        "seed": int(SEED),
        "model_ckpt": MODEL_CKPT,
        "model_tag": MODEL_TAG,
        "run_type": "finetuned_single_run",
        "max_len_train": int(MAX_LEN_TRAIN),
        "use_sliding": USE_SLIDING,
        "max_len_sw": int(MAX_LEN_SW),
        "stride_sw": int(STRIDE_SW),
        "agg_method": AGG_METHOD,
        "learning_rate": float(2e-5),
        "train_batch_size": int(16),
        "eval_batch_size": int(32),
        "epochs": int(4),
        "weight_decay": float(0.01),
        "selection_metric": "test/f1_macro_sliding" if USE_SLIDING else "test/f1_macro_simple",
    },
    tags=["finetune", "single_run", "rouberta", "sliding" if USE_SLIDING else "simple"],
    reinit="finish_previous",
)

print("RUN:", wandb.run.name, "ID:", wandb.run.id, "USE_SLIDING:", USE_SLIDING, "AGG:", AGG_METHOD)

# =========================================
# 3) Utils
# =========================================
def softmax_stable(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

@torch.no_grad()
def predict_proba_windows(
    text: str,
    model,
    tokenizer,
    max_len: int = 128,
    stride: int = 64,
    device=None
):
    model.eval()
    if device is None:
        device = model.device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    if len(input_ids) > 0 and isinstance(input_ids[0], int):
        input_ids = [input_ids]
        attention_mask = [attention_mask]

    max_batch_len = max(len(x) for x in input_ids)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

    def pad(seq, pad_value):
        return seq + [pad_value] * (max_batch_len - len(seq))

    input_ids = torch.tensor(
        [pad(x, pad_id) for x in input_ids],
        dtype=torch.long,
        device=device
    )
    attention_mask = torch.tensor(
        [pad(x, 0) for x in attention_mask],
        dtype=torch.long,
        device=device
    )

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    return probs, len(probs)

@torch.no_grad()
def predict_text_truncated(text, model, tokenizer, id2label, max_len=128, device=None):
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        return_attention_mask=True,
        padding="max_length",
        return_tensors="pt",
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()[0]

    pred_id = int(np.argmax(probs))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs))
    n_win = 1
    votes_counts = None

    return pred_label, conf, probs, n_win, votes_counts

def aggregate_probs(probs_windows: np.ndarray, method: str = "mean"):
    if method == "mean":
        return probs_windows.mean(axis=0)
    if method == "max":
        return probs_windows.max(axis=0)
    if method == "vote":
        votes = np.argmax(probs_windows, axis=1)
        counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(float)
        return counts / counts.sum()
    raise ValueError("method debe ser 'mean', 'max' o 'vote'")

def predict_text_with_sliding(text, model, tokenizer, id2label,
                              max_len=128, stride=64, agg="mean"):
    """
    Retorna:
      pred_label (str),
      conf (float),
      probs_agg (np.array),
      n_win (int),
      votes_counts (np.array int) o None
    """
    probs_windows, n_win = predict_proba_windows(
        text=text,
        model=model,
        tokenizer=tokenizer,
        max_len=max_len,
        stride=stride
    )
    votes_counts = None

    if agg == "vote":
        votes = np.argmax(probs_windows, axis=1)
        votes_counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(int)
        max_count = votes_counts.max()
        tied = np.where(votes_counts == max_count)[0]

        if len(tied) == 1:
            pred_id = int(tied[0])
        else:
            mean_probs = probs_windows.mean(axis=0)
            pred_id = int(tied[np.argmax(mean_probs[tied])])

        probs_agg = votes_counts.astype(float) / votes_counts.sum()
        conf = float(probs_agg[pred_id])
        pred_label = id2label[pred_id]
        return pred_label, conf, probs_agg, n_win, votes_counts

    probs_agg = aggregate_probs(probs_windows, method=agg)
    pred_id = int(np.argmax(probs_agg))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs_agg))
    return pred_label, conf, probs_agg, n_win, votes_counts

# =========================================
# 4) Datos
# =========================================
df_lab = df_limpio[df_limpio["sentimiento"].notna()].copy()
df_lab["text"] = df_lab["intervencion"].astype(str)

valid_labels = ["N", "Neu", "P"]
df_lab = df_lab[df_lab["sentimiento"].isin(valid_labels)].copy()

label_list = ["N", "Neu", "P"]
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
df_lab["label"] = df_lab["sentimiento"].map(label2id)

# Split 80/10/10
train_df, temp_df = train_test_split(
    df_lab,
    test_size=0.20,
    random_state=SEED,
    stratify=df_lab["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

wandb.log({
    "data/n_train": len(train_df),
    "data/n_val": len(val_df),
    "data/n_test": len(test_df),
})

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# =========================================
# 5) Tokenize
# =========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize_train(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN_TRAIN)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

ds = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})
ds_tok = ds.map(tokenize_train, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================================
# 6) Model
# =========================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# =========================================
# 7) Metrics para Trainer
# =========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# =========================================
# 8) Trainer + Train
# =========================================
use_fp16 = torch.cuda.is_available()
OUTPUT_DIR = f"{MODEL_TAG}_finetuned_{wandb.run.id}"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    fp16=use_fp16,
    report_to="wandb",
    run_name=wandb.run.name,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

best_model = trainer.model

# =========================================
# 9) VALIDATION simple — referencia, no selección final
# =========================================
pred_val_simple = trainer.predict(ds_tok["validation"])
logits_val_simple = pred_val_simple.predictions
y_true_val_simple = pred_val_simple.label_ids
y_pred_val_simple = np.argmax(logits_val_simple, axis=-1)

eval_simple_metrics, eval_simple_report = build_metrics_dict(
    y_true=y_true_val_simple,
    y_pred=y_pred_val_simple,
    label_list=label_list,
    label2id=label2id,
    split="eval",
    variant="simple"
)

wandb.log(eval_simple_metrics)

print("\nVALIDATION SIMPLE")
print(f"  Accuracy: {eval_simple_metrics['eval/accuracy_simple']:.4f}")
print(f"  F1-macro: {eval_simple_metrics['eval/f1_macro_simple']:.4f}")
print(classification_report(
    y_true_val_simple,
    y_pred_val_simple,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 10) VALIDATION sliding — referencia, no selección final
# =========================================
if USE_SLIDING:
    y_true_val_sliding = val_df["label"].to_numpy()
    y_pred_val_sw = []
    nwin_val = []

    for txt in val_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_val_sw.append(label2id[lab])
        nwin_val.append(n_win)

    y_pred_val_sw = np.array(y_pred_val_sw)

    eval_sliding_metrics, eval_sliding_report = build_metrics_dict(
        y_true=y_true_val_sliding,
        y_pred=y_pred_val_sw,
        label_list=label_list,
        label2id=label2id,
        split="eval",
        variant="sliding"
    )

    eval_sliding_metrics.update({
        "eval/avg_windows_sliding": float(np.mean(nwin_val)),
        "eval/max_windows_sliding": float(np.max(nwin_val)),
    })

    wandb.log(eval_sliding_metrics)

    print("\nVALIDATION SLIDING")
    print(f"  Accuracy: {eval_sliding_metrics['eval/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {eval_sliding_metrics['eval/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true_val_sliding,
        y_pred_val_sw,
        target_names=label_list,
        zero_division=0
    ))

# =========================================
# 11) TEST simple — resultado final comparable
# =========================================
pred_test_simple = trainer.predict(ds_tok["test"])
logits_test_simple = pred_test_simple.predictions
y_true = pred_test_simple.label_ids
y_pred = np.argmax(logits_test_simple, axis=-1)
probs = softmax_stable(logits_test_simple, axis=1)

true_label_txt = np.array([id2label[i] for i in y_true])
pred_label_txt = np.array([id2label[i] for i in y_pred])

test_meta = test_df.reset_index(drop=True).copy()
test_meta["true_label"] = true_label_txt
test_meta["pred_label"] = pred_label_txt
test_meta["conf_pred"] = probs.max(axis=1)
test_meta["proba_N"] = probs[:, label2id["N"]]
test_meta["proba_Neu"] = probs[:, label2id["Neu"]]
test_meta["proba_P"] = probs[:, label2id["P"]]
test_meta["proba_vector"] = probs.tolist()

test_simple_metrics, test_simple_report = build_metrics_dict(
    y_true=y_true,
    y_pred=y_pred,
    label_list=label_list,
    label2id=label2id,
    split="test",
    variant="simple"
)

wandb.log(test_simple_metrics)

print("\nTEST SIMPLE")
print(f"  Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f}")
print(f"  F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f}")
print(classification_report(
    y_true,
    y_pred,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 12) TEST sliding — resultado final comparable
# =========================================
if USE_SLIDING:
    y_pred_sw = []
    conf_sw = []
    nwin_sw = []
    votes_N_list, votes_Neu_list, votes_P_list = [], [], []

    for txt in test_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_sw.append(label2id[lab])
        conf_sw.append(conf)
        nwin_sw.append(n_win)

        if votes_counts is None:
            votes_N_list.append(np.nan)
            votes_Neu_list.append(np.nan)
            votes_P_list.append(np.nan)
        else:
            votes_N_list.append(int(votes_counts[label2id["N"]]))
            votes_Neu_list.append(int(votes_counts[label2id["Neu"]]))
            votes_P_list.append(int(votes_counts[label2id["P"]]))

    y_pred_sw = np.array(y_pred_sw)

    assert len(test_df) == len(y_true) == len(y_pred_sw), "Mismatch en longitudes de test"

    test_meta["pred_label_sw"] = [id2label[i] for i in y_pred_sw]
    test_meta["conf_sw"] = conf_sw
    test_meta["n_windows"] = nwin_sw
    test_meta["votes_N"] = votes_N_list
    test_meta["votes_Neu"] = votes_Neu_list
    test_meta["votes_P"] = votes_P_list

    test_sliding_metrics, test_sliding_report = build_metrics_dict(
        y_true=y_true,
        y_pred=y_pred_sw,
        label_list=label_list,
        label2id=label2id,
        split="test",
        variant="sliding"
    )

    test_sliding_metrics.update({
        "test/avg_windows_sliding": float(np.mean(nwin_sw)),
        "test/max_windows_sliding": float(np.max(nwin_sw)),
    })

    wandb.log(test_sliding_metrics)

    print("\nTEST SLIDING")
    print(f"  Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true,
        y_pred_sw,
        target_names=label_list,
        zero_division=0
    ))

else:
    test_meta["pred_label_sw"] = np.nan
    test_meta["conf_sw"] = np.nan
    test_meta["n_windows"] = np.nan
    test_meta["votes_N"] = np.nan
    test_meta["votes_Neu"] = np.nan
    test_meta["votes_P"] = np.nan

# =========================================
# 13) Export Excel (test_all + errores simple + errores sliding)
# =========================================
cols_excel = [
    "id_intervencion", "file_name", "n_legislatura", "cuerpo", "fecha",
    "locutor", "encabezado", "intervencion", "n_palabras", "n_caracteres",
    "año", "sexo_locutor", "apellido_locutor_norm", "partido_imputado",
    "sentimiento", "text",
    "true_label",
    "pred_label", "conf_pred", "proba_N", "proba_Neu", "proba_P", "proba_vector",
    "pred_label_sw", "conf_sw", "n_windows", "votes_N", "votes_Neu", "votes_P",
]
cols_excel = [c for c in cols_excel if c in test_meta.columns]

df_errors_simple = test_meta[test_meta["true_label"] != test_meta["pred_label"]].copy()
df_errors_sliding = (
    test_meta[test_meta["true_label"] != test_meta["pred_label_sw"]].copy()
    if USE_SLIDING else pd.DataFrame()
)
df_corrige_sliding = (
    test_meta[
        (test_meta["true_label"] != test_meta["pred_label"]) &
        (test_meta["true_label"] == test_meta["pred_label_sw"])
    ].copy()
    if USE_SLIDING else pd.DataFrame()
)

out_path = f"analisis_{RUN_NAME}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    test_meta[cols_excel].to_excel(writer, index=False, sheet_name="test_all")
    df_errors_simple[cols_excel].to_excel(writer, index=False, sheet_name="errores_simple")
    if USE_SLIDING:
        df_errors_sliding[cols_excel].to_excel(writer, index=False, sheet_name="errores_sliding")
        df_corrige_sliding[cols_excel].to_excel(writer, index=False, sheet_name="corrige_sliding")

print("Excel generado:", out_path)
print(f"  test total:        {len(test_meta)}")
print(f"  errores simple:    {len(df_errors_simple)}")
if USE_SLIDING:
    print(f"  errores sliding:   {len(df_errors_sliding)}")
    print(f"  corrige sliding:   {len(df_corrige_sliding)}")

print("\n=== TEST SIMPLE ===")
print(classification_report(y_true, y_pred, target_names=label_list, zero_division=0))

if USE_SLIDING:
    print("\n=== TEST SLIDING ===")
    print(classification_report(y_true, y_pred_sw, target_names=label_list, zero_division=0))

print("\n" + "=" * 60)
print("RESUMEN FINAL — TEST")
print(f"Simple  | Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f} | "
      f"F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f} | "
      f"F1-weighted: {test_simple_metrics['test/f1_weighted_simple']:.4f}")

if USE_SLIDING:
    print(f"Sliding | Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f} | "
          f"F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f} | "
          f"F1-weighted: {test_sliding_metrics['test/f1_weighted_sliding']:.4f}")
print("=" * 60)

wandb.finish()

RUN: rouberta_slide_1899_vote ID: eene9mwj USE_SLIDING: True AGG: vote
Train: 979 Val: 122 Test: 123


Map:   0%|          | 0/979 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


VALIDATION SIMPLE
  Accuracy: 0.8033
  F1-macro: 0.8000
              precision    recall  f1-score   support

           N       0.74      0.85      0.80        41
         Neu       0.83      0.84      0.84        45
           P       0.86      0.69      0.77        36

    accuracy                           0.80       122
   macro avg       0.81      0.80      0.80       122
weighted avg       0.81      0.80      0.80       122


VALIDATION SLIDING
  Accuracy: 0.7377
  F1-macro: 0.6924
              precision    recall  f1-score   support

           N       0.77      0.90      0.83        41
         Neu       0.67      0.93      0.78        45
           P       1.00      0.31      0.47        36

    accuracy                           0.74       122
   macro avg       0.81      0.71      0.69       122
weighted avg       0.80      0.74      0.70       122




TEST SIMPLE
  Accuracy: 0.7154
  F1-macro: 0.7036
              precision    recall  f1-score   support

           N       0.66      0.85      0.74        41
         Neu       0.83      0.74      0.78        46
           P       0.66      0.53      0.58        36

    accuracy                           0.72       123
   macro avg       0.71      0.71      0.70       123
weighted avg       0.72      0.72      0.71       123


TEST SLIDING
  Accuracy: 0.6585
  F1-macro: 0.6126
              precision    recall  f1-score   support

           N       0.69      0.83      0.76        41
         Neu       0.59      0.83      0.69        46
           P       0.90      0.25      0.39        36

    accuracy                           0.66       123
   macro avg       0.73      0.64      0.61       123
weighted avg       0.72      0.66      0.62       123

Excel generado: analisis_rouberta_slide_1899_vote.xlsx
  test total:        123
  errores simple:    35
  errores sliding:   42
  corri

data/n_test,▁
data/n_train,▁
data/n_val,▁
eval/accuracy,▁█▆▆
eval/accuracy_simple,▁
eval/accuracy_sliding,▁
eval/avg_windows_sliding,▁
eval/class_N_accuracy_ovr_simple,▁
eval/class_N_accuracy_ovr_sliding,▁
eval/class_N_f1_simple,▁
+88,...


## Slide + Mean

In [ ]:
# =========================================
# 0) W&B + Config (antes de todo)
# =========================================
import os
import random
import numpy as np
import pandas as pd
import torch
import wandb

WANDB_PROJECT = "Tesis ORT"
WANDB_ENTITY  = "fbenedetti97-universidad-ort-uruguay"

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["WANDB_ENTITY"] = WANDB_ENTITY

if wandb.run is not None:
    wandb.finish()

USE_SLIDING = True
AGG_METHOD  = "mean"   # "mean" / "vote"

# =========================================
# 1) Imports + Config
# =========================================
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

SEED       = 1899
MODEL_CKPT = "pln-udelar/rouberta-base-uy22-cased"
MODEL_TAG  = "rouberta"

RUN_NAME   = (
    f"{MODEL_TAG}_base_{SEED}"
    if not USE_SLIDING
    else f"{MODEL_TAG}_slide_{SEED}_{AGG_METHOD}"
)

MAX_LEN_TRAIN = 128
MAX_LEN_SW    = 128
STRIDE_SW     = 96

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================
# 2) Init W&B
# =========================================
run = wandb.init(
    project=WANDB_PROJECT,
    # entity=WANDB_ENTITY,
    name=RUN_NAME,
    config={
        "seed": int(SEED),
        "model_ckpt": MODEL_CKPT,
        "model_tag": MODEL_TAG,
        "run_type": "finetuned_single_run",
        "max_len_train": int(MAX_LEN_TRAIN),
        "use_sliding": USE_SLIDING,
        "max_len_sw": int(MAX_LEN_SW),
        "stride_sw": int(STRIDE_SW),
        "agg_method": AGG_METHOD,
        "learning_rate": float(2e-5),
        "train_batch_size": int(16),
        "eval_batch_size": int(32),
        "epochs": int(4),
        "weight_decay": float(0.01),
        "selection_metric": "test/f1_macro_sliding" if USE_SLIDING else "test/f1_macro_simple",
    },
    tags=["finetune", "single_run", "rouberta", "sliding" if USE_SLIDING else "simple"],
    reinit="finish_previous",
)

print("RUN:", wandb.run.name, "ID:", wandb.run.id, "USE_SLIDING:", USE_SLIDING, "AGG:", AGG_METHOD)

# =========================================
# 3) Utils
# =========================================
def softmax_stable(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

@torch.no_grad()
def predict_proba_windows(
    text: str,
    model,
    tokenizer,
    max_len: int = 128,
    stride: int = 64,
    device=None
):
    model.eval()
    if device is None:
        device = model.device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    if len(input_ids) > 0 and isinstance(input_ids[0], int):
        input_ids = [input_ids]
        attention_mask = [attention_mask]

    max_batch_len = max(len(x) for x in input_ids)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

    def pad(seq, pad_value):
        return seq + [pad_value] * (max_batch_len - len(seq))

    input_ids = torch.tensor(
        [pad(x, pad_id) for x in input_ids],
        dtype=torch.long,
        device=device
    )
    attention_mask = torch.tensor(
        [pad(x, 0) for x in attention_mask],
        dtype=torch.long,
        device=device
    )

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    return probs, len(probs)

@torch.no_grad()
def predict_text_truncated(text, model, tokenizer, id2label, max_len=128, device=None):
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        return_attention_mask=True,
        padding="max_length",
        return_tensors="pt",
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()[0]

    pred_id = int(np.argmax(probs))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs))
    n_win = 1
    votes_counts = None

    return pred_label, conf, probs, n_win, votes_counts

def aggregate_probs(probs_windows: np.ndarray, method: str = "mean"):
    if method == "mean":
        return probs_windows.mean(axis=0)
    if method == "max":
        return probs_windows.max(axis=0)
    if method == "vote":
        votes = np.argmax(probs_windows, axis=1)
        counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(float)
        return counts / counts.sum()
    raise ValueError("method debe ser 'mean', 'max' o 'vote'")

def predict_text_with_sliding(text, model, tokenizer, id2label,
                              max_len=128, stride=64, agg="mean"):
    """
    Retorna:
      pred_label (str),
      conf (float),
      probs_agg (np.array),
      n_win (int),
      votes_counts (np.array int) o None
    """
    probs_windows, n_win = predict_proba_windows(
        text=text,
        model=model,
        tokenizer=tokenizer,
        max_len=max_len,
        stride=stride
    )
    votes_counts = None

    if agg == "vote":
        votes = np.argmax(probs_windows, axis=1)
        votes_counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(int)
        max_count = votes_counts.max()
        tied = np.where(votes_counts == max_count)[0]

        if len(tied) == 1:
            pred_id = int(tied[0])
        else:
            mean_probs = probs_windows.mean(axis=0)
            pred_id = int(tied[np.argmax(mean_probs[tied])])

        probs_agg = votes_counts.astype(float) / votes_counts.sum()
        conf = float(probs_agg[pred_id])
        pred_label = id2label[pred_id]
        return pred_label, conf, probs_agg, n_win, votes_counts

    probs_agg = aggregate_probs(probs_windows, method=agg)
    pred_id = int(np.argmax(probs_agg))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs_agg))
    return pred_label, conf, probs_agg, n_win, votes_counts

# =========================================
# 4) Datos
# =========================================
df_lab = df_limpio[df_limpio["sentimiento"].notna()].copy()
df_lab["text"] = df_lab["intervencion"].astype(str)

valid_labels = ["N", "Neu", "P"]
df_lab = df_lab[df_lab["sentimiento"].isin(valid_labels)].copy()

label_list = ["N", "Neu", "P"]
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
df_lab["label"] = df_lab["sentimiento"].map(label2id)

# Split 80/10/10
train_df, temp_df = train_test_split(
    df_lab,
    test_size=0.20,
    random_state=SEED,
    stratify=df_lab["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

wandb.log({
    "data/n_train": len(train_df),
    "data/n_val": len(val_df),
    "data/n_test": len(test_df),
})

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# =========================================
# 5) Tokenize
# =========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize_train(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN_TRAIN)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

ds = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})
ds_tok = ds.map(tokenize_train, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================================
# 6) Model
# =========================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# =========================================
# 7) Metrics para Trainer
# =========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# =========================================
# 8) Trainer + Train
# =========================================
use_fp16 = torch.cuda.is_available()
OUTPUT_DIR = f"{MODEL_TAG}_finetuned_{wandb.run.id}"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    fp16=use_fp16,
    report_to="wandb",
    run_name=wandb.run.name,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

best_model = trainer.model

# =========================================
# 9) VALIDATION simple — referencia, no selección final
# =========================================
pred_val_simple = trainer.predict(ds_tok["validation"])
logits_val_simple = pred_val_simple.predictions
y_true_val_simple = pred_val_simple.label_ids
y_pred_val_simple = np.argmax(logits_val_simple, axis=-1)

eval_simple_metrics, eval_simple_report = build_metrics_dict(
    y_true=y_true_val_simple,
    y_pred=y_pred_val_simple,
    label_list=label_list,
    label2id=label2id,
    split="eval",
    variant="simple"
)

wandb.log(eval_simple_metrics)

print("\nVALIDATION SIMPLE")
print(f"  Accuracy: {eval_simple_metrics['eval/accuracy_simple']:.4f}")
print(f"  F1-macro: {eval_simple_metrics['eval/f1_macro_simple']:.4f}")
print(classification_report(
    y_true_val_simple,
    y_pred_val_simple,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 10) VALIDATION sliding — referencia, no selección final
# =========================================
if USE_SLIDING:
    y_true_val_sliding = val_df["label"].to_numpy()
    y_pred_val_sw = []
    nwin_val = []

    for txt in val_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_val_sw.append(label2id[lab])
        nwin_val.append(n_win)

    y_pred_val_sw = np.array(y_pred_val_sw)

    eval_sliding_metrics, eval_sliding_report = build_metrics_dict(
        y_true=y_true_val_sliding,
        y_pred=y_pred_val_sw,
        label_list=label_list,
        label2id=label2id,
        split="eval",
        variant="sliding"
    )

    eval_sliding_metrics.update({
        "eval/avg_windows_sliding": float(np.mean(nwin_val)),
        "eval/max_windows_sliding": float(np.max(nwin_val)),
    })

    wandb.log(eval_sliding_metrics)

    print("\nVALIDATION SLIDING")
    print(f"  Accuracy: {eval_sliding_metrics['eval/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {eval_sliding_metrics['eval/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true_val_sliding,
        y_pred_val_sw,
        target_names=label_list,
        zero_division=0
    ))

# =========================================
# 11) TEST simple — resultado final comparable
# =========================================
pred_test_simple = trainer.predict(ds_tok["test"])
logits_test_simple = pred_test_simple.predictions
y_true = pred_test_simple.label_ids
y_pred = np.argmax(logits_test_simple, axis=-1)
probs = softmax_stable(logits_test_simple, axis=1)

true_label_txt = np.array([id2label[i] for i in y_true])
pred_label_txt = np.array([id2label[i] for i in y_pred])

test_meta = test_df.reset_index(drop=True).copy()
test_meta["true_label"] = true_label_txt
test_meta["pred_label"] = pred_label_txt
test_meta["conf_pred"] = probs.max(axis=1)
test_meta["proba_N"] = probs[:, label2id["N"]]
test_meta["proba_Neu"] = probs[:, label2id["Neu"]]
test_meta["proba_P"] = probs[:, label2id["P"]]
test_meta["proba_vector"] = probs.tolist()

test_simple_metrics, test_simple_report = build_metrics_dict(
    y_true=y_true,
    y_pred=y_pred,
    label_list=label_list,
    label2id=label2id,
    split="test",
    variant="simple"
)

wandb.log(test_simple_metrics)

print("\nTEST SIMPLE")
print(f"  Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f}")
print(f"  F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f}")
print(classification_report(
    y_true,
    y_pred,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 12) TEST sliding — resultado final comparable
# =========================================
if USE_SLIDING:
    y_pred_sw = []
    conf_sw = []
    nwin_sw = []
    votes_N_list, votes_Neu_list, votes_P_list = [], [], []

    for txt in test_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_sw.append(label2id[lab])
        conf_sw.append(conf)
        nwin_sw.append(n_win)

        if votes_counts is None:
            votes_N_list.append(np.nan)
            votes_Neu_list.append(np.nan)
            votes_P_list.append(np.nan)
        else:
            votes_N_list.append(int(votes_counts[label2id["N"]]))
            votes_Neu_list.append(int(votes_counts[label2id["Neu"]]))
            votes_P_list.append(int(votes_counts[label2id["P"]]))

    y_pred_sw = np.array(y_pred_sw)

    assert len(test_df) == len(y_true) == len(y_pred_sw), "Mismatch en longitudes de test"

    test_meta["pred_label_sw"] = [id2label[i] for i in y_pred_sw]
    test_meta["conf_sw"] = conf_sw
    test_meta["n_windows"] = nwin_sw
    test_meta["votes_N"] = votes_N_list
    test_meta["votes_Neu"] = votes_Neu_list
    test_meta["votes_P"] = votes_P_list

    test_sliding_metrics, test_sliding_report = build_metrics_dict(
        y_true=y_true,
        y_pred=y_pred_sw,
        label_list=label_list,
        label2id=label2id,
        split="test",
        variant="sliding"
    )

    test_sliding_metrics.update({
        "test/avg_windows_sliding": float(np.mean(nwin_sw)),
        "test/max_windows_sliding": float(np.max(nwin_sw)),
    })

    wandb.log(test_sliding_metrics)

    print("\nTEST SLIDING")
    print(f"  Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true,
        y_pred_sw,
        target_names=label_list,
        zero_division=0
    ))

else:
    test_meta["pred_label_sw"] = np.nan
    test_meta["conf_sw"] = np.nan
    test_meta["n_windows"] = np.nan
    test_meta["votes_N"] = np.nan
    test_meta["votes_Neu"] = np.nan
    test_meta["votes_P"] = np.nan

# =========================================
# 13) Export Excel (test_all + errores simple + errores sliding)
# =========================================
cols_excel = [
    "id_intervencion", "file_name", "n_legislatura", "cuerpo", "fecha",
    "locutor", "encabezado", "intervencion", "n_palabras", "n_caracteres",
    "año", "sexo_locutor", "apellido_locutor_norm", "partido_imputado",
    "sentimiento", "text",
    "true_label",
    "pred_label", "conf_pred", "proba_N", "proba_Neu", "proba_P", "proba_vector",
    "pred_label_sw", "conf_sw", "n_windows", "votes_N", "votes_Neu", "votes_P",
]
cols_excel = [c for c in cols_excel if c in test_meta.columns]

df_errors_simple = test_meta[test_meta["true_label"] != test_meta["pred_label"]].copy()
df_errors_sliding = (
    test_meta[test_meta["true_label"] != test_meta["pred_label_sw"]].copy()
    if USE_SLIDING else pd.DataFrame()
)
df_corrige_sliding = (
    test_meta[
        (test_meta["true_label"] != test_meta["pred_label"]) &
        (test_meta["true_label"] == test_meta["pred_label_sw"])
    ].copy()
    if USE_SLIDING else pd.DataFrame()
)

out_path = f"analisis_{RUN_NAME}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    test_meta[cols_excel].to_excel(writer, index=False, sheet_name="test_all")
    df_errors_simple[cols_excel].to_excel(writer, index=False, sheet_name="errores_simple")
    if USE_SLIDING:
        df_errors_sliding[cols_excel].to_excel(writer, index=False, sheet_name="errores_sliding")
        df_corrige_sliding[cols_excel].to_excel(writer, index=False, sheet_name="corrige_sliding")

print("Excel generado:", out_path)
print(f"  test total:        {len(test_meta)}")
print(f"  errores simple:    {len(df_errors_simple)}")
if USE_SLIDING:
    print(f"  errores sliding:   {len(df_errors_sliding)}")
    print(f"  corrige sliding:   {len(df_corrige_sliding)}")

print("\n=== TEST SIMPLE ===")
print(classification_report(y_true, y_pred, target_names=label_list, zero_division=0))

if USE_SLIDING:
    print("\n=== TEST SLIDING ===")
    print(classification_report(y_true, y_pred_sw, target_names=label_list, zero_division=0))

print("\n" + "=" * 60)
print("RESUMEN FINAL — TEST")
print(f"Simple  | Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f} | "
      f"F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f} | "
      f"F1-weighted: {test_simple_metrics['test/f1_weighted_simple']:.4f}")

if USE_SLIDING:
    print(f"Sliding | Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f} | "
          f"F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f} | "
          f"F1-weighted: {test_sliding_metrics['test/f1_weighted_sliding']:.4f}")
print("=" * 60)

wandb.finish()

RUN: rouberta_slide_1899_mean ID: woda6hxd USE_SLIDING: True AGG: mean
Train: 979 Val: 122 Test: 123


Map:   0%|          | 0/979 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


VALIDATION SIMPLE
  Accuracy: 0.8033
  F1-macro: 0.8000
              precision    recall  f1-score   support

           N       0.74      0.85      0.80        41
         Neu       0.83      0.84      0.84        45
           P       0.86      0.69      0.77        36

    accuracy                           0.80       122
   macro avg       0.81      0.80      0.80       122
weighted avg       0.81      0.80      0.80       122


VALIDATION SLIDING
  Accuracy: 0.7459
  F1-macro: 0.7093
              precision    recall  f1-score   support

           N       0.75      0.93      0.83        41
         Neu       0.71      0.89      0.79        45
           P       0.87      0.36      0.51        36

    accuracy                           0.75       122
   macro avg       0.78      0.73      0.71       122
weighted avg       0.77      0.75      0.72       122




TEST SIMPLE
  Accuracy: 0.7154
  F1-macro: 0.7036
              precision    recall  f1-score   support

           N       0.66      0.85      0.74        41
         Neu       0.83      0.74      0.78        46
           P       0.66      0.53      0.58        36

    accuracy                           0.72       123
   macro avg       0.71      0.71      0.70       123
weighted avg       0.72      0.72      0.71       123


TEST SLIDING
  Accuracy: 0.7073
  F1-macro: 0.6870
              precision    recall  f1-score   support

           N       0.71      0.83      0.76        41
         Neu       0.66      0.83      0.73        46
           P       0.88      0.42      0.57        36

    accuracy                           0.71       123
   macro avg       0.75      0.69      0.69       123
weighted avg       0.74      0.71      0.69       123

Excel generado: analisis_rouberta_slide_1899_mean.xlsx
  test total:        123
  errores simple:    35
  errores sliding:   36
  corri

data/n_test,▁
data/n_train,▁
data/n_val,▁
eval/accuracy,▁█▆▆
eval/accuracy_simple,▁
eval/accuracy_sliding,▁
eval/avg_windows_sliding,▁
eval/class_N_accuracy_ovr_simple,▁
eval/class_N_accuracy_ovr_sliding,▁
eval/class_N_f1_simple,▁
+88,...


## Rouberta SEED 1405


## Truncado

In [ ]:
# =========================================
# 0) W&B + Config (antes de todo)
# =========================================
import os
import random
import numpy as np
import pandas as pd
import torch
import wandb

WANDB_PROJECT = "Tesis ORT"
WANDB_ENTITY  = "fbenedetti97-universidad-ort-uruguay"

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["WANDB_ENTITY"] = WANDB_ENTITY

if wandb.run is not None:
    wandb.finish()

USE_SLIDING = False
AGG_METHOD  = "mean"   # "mean" / "vote"

# =========================================
# 1) Imports + Config
# =========================================
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

SEED       = 1405
MODEL_CKPT = "pln-udelar/rouberta-base-uy22-cased"
MODEL_TAG  = "rouberta"

RUN_NAME   = (
    f"{MODEL_TAG}_base_{SEED}"
    if not USE_SLIDING
    else f"{MODEL_TAG}_slide_{SEED}_{AGG_METHOD}"
)

MAX_LEN_TRAIN = 128
MAX_LEN_SW    = 128
STRIDE_SW     = 96

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================
# 2) Init W&B
# =========================================
run = wandb.init(
    project=WANDB_PROJECT,
    # entity=WANDB_ENTITY,
    name=RUN_NAME,
    config={
        "seed": int(SEED),
        "model_ckpt": MODEL_CKPT,
        "model_tag": MODEL_TAG,
        "run_type": "finetuned_single_run",
        "max_len_train": int(MAX_LEN_TRAIN),
        "use_sliding": USE_SLIDING,
        "max_len_sw": int(MAX_LEN_SW),
        "stride_sw": int(STRIDE_SW),
        "agg_method": AGG_METHOD,
        "learning_rate": float(2e-5),
        "train_batch_size": int(16),
        "eval_batch_size": int(32),
        "epochs": int(4),
        "weight_decay": float(0.01),
        "selection_metric": "test/f1_macro_sliding" if USE_SLIDING else "test/f1_macro_simple",
    },
    tags=["finetune", "single_run", "rouberta", "sliding" if USE_SLIDING else "simple"],
    reinit="finish_previous",
)

print("RUN:", wandb.run.name, "ID:", wandb.run.id, "USE_SLIDING:", USE_SLIDING, "AGG:", AGG_METHOD)

# =========================================
# 3) Utils
# =========================================
def softmax_stable(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

@torch.no_grad()
def predict_proba_windows(
    text: str,
    model,
    tokenizer,
    max_len: int = 128,
    stride: int = 64,
    device=None
):
    model.eval()
    if device is None:
        device = model.device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    if len(input_ids) > 0 and isinstance(input_ids[0], int):
        input_ids = [input_ids]
        attention_mask = [attention_mask]

    max_batch_len = max(len(x) for x in input_ids)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

    def pad(seq, pad_value):
        return seq + [pad_value] * (max_batch_len - len(seq))

    input_ids = torch.tensor(
        [pad(x, pad_id) for x in input_ids],
        dtype=torch.long,
        device=device
    )
    attention_mask = torch.tensor(
        [pad(x, 0) for x in attention_mask],
        dtype=torch.long,
        device=device
    )

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    return probs, len(probs)

@torch.no_grad()
def predict_text_truncated(text, model, tokenizer, id2label, max_len=128, device=None):
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        return_attention_mask=True,
        padding="max_length",
        return_tensors="pt",
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()[0]

    pred_id = int(np.argmax(probs))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs))
    n_win = 1
    votes_counts = None

    return pred_label, conf, probs, n_win, votes_counts

def aggregate_probs(probs_windows: np.ndarray, method: str = "mean"):
    if method == "mean":
        return probs_windows.mean(axis=0)
    if method == "max":
        return probs_windows.max(axis=0)
    if method == "vote":
        votes = np.argmax(probs_windows, axis=1)
        counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(float)
        return counts / counts.sum()
    raise ValueError("method debe ser 'mean', 'max' o 'vote'")

def predict_text_with_sliding(text, model, tokenizer, id2label,
                              max_len=128, stride=64, agg="mean"):
    """
    Retorna:
      pred_label (str),
      conf (float),
      probs_agg (np.array),
      n_win (int),
      votes_counts (np.array int) o None
    """
    probs_windows, n_win = predict_proba_windows(
        text=text,
        model=model,
        tokenizer=tokenizer,
        max_len=max_len,
        stride=stride
    )
    votes_counts = None

    if agg == "vote":
        votes = np.argmax(probs_windows, axis=1)
        votes_counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(int)
        max_count = votes_counts.max()
        tied = np.where(votes_counts == max_count)[0]

        if len(tied) == 1:
            pred_id = int(tied[0])
        else:
            mean_probs = probs_windows.mean(axis=0)
            pred_id = int(tied[np.argmax(mean_probs[tied])])

        probs_agg = votes_counts.astype(float) / votes_counts.sum()
        conf = float(probs_agg[pred_id])
        pred_label = id2label[pred_id]
        return pred_label, conf, probs_agg, n_win, votes_counts

    probs_agg = aggregate_probs(probs_windows, method=agg)
    pred_id = int(np.argmax(probs_agg))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs_agg))
    return pred_label, conf, probs_agg, n_win, votes_counts

# =========================================
# 4) Datos
# =========================================
df_lab = df_limpio[df_limpio["sentimiento"].notna()].copy()
df_lab["text"] = df_lab["intervencion"].astype(str)

valid_labels = ["N", "Neu", "P"]
df_lab = df_lab[df_lab["sentimiento"].isin(valid_labels)].copy()

label_list = ["N", "Neu", "P"]
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
df_lab["label"] = df_lab["sentimiento"].map(label2id)

# Split 80/10/10
train_df, temp_df = train_test_split(
    df_lab,
    test_size=0.20,
    random_state=SEED,
    stratify=df_lab["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

wandb.log({
    "data/n_train": len(train_df),
    "data/n_val": len(val_df),
    "data/n_test": len(test_df),
})

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# =========================================
# 5) Tokenize
# =========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize_train(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN_TRAIN)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

ds = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})
ds_tok = ds.map(tokenize_train, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================================
# 6) Model
# =========================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# =========================================
# 7) Metrics para Trainer
# =========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# =========================================
# 8) Trainer + Train
# =========================================
use_fp16 = torch.cuda.is_available()
OUTPUT_DIR = f"{MODEL_TAG}_finetuned_{wandb.run.id}"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    fp16=use_fp16,
    report_to="wandb",
    run_name=wandb.run.name,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

best_model = trainer.model

# =========================================
# 9) VALIDATION simple — referencia, no selección final
# =========================================
pred_val_simple = trainer.predict(ds_tok["validation"])
logits_val_simple = pred_val_simple.predictions
y_true_val_simple = pred_val_simple.label_ids
y_pred_val_simple = np.argmax(logits_val_simple, axis=-1)

eval_simple_metrics, eval_simple_report = build_metrics_dict(
    y_true=y_true_val_simple,
    y_pred=y_pred_val_simple,
    label_list=label_list,
    label2id=label2id,
    split="eval",
    variant="simple"
)

wandb.log(eval_simple_metrics)

print("\nVALIDATION SIMPLE")
print(f"  Accuracy: {eval_simple_metrics['eval/accuracy_simple']:.4f}")
print(f"  F1-macro: {eval_simple_metrics['eval/f1_macro_simple']:.4f}")
print(classification_report(
    y_true_val_simple,
    y_pred_val_simple,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 10) VALIDATION sliding — referencia, no selección final
# =========================================
if USE_SLIDING:
    y_true_val_sliding = val_df["label"].to_numpy()
    y_pred_val_sw = []
    nwin_val = []

    for txt in val_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_val_sw.append(label2id[lab])
        nwin_val.append(n_win)

    y_pred_val_sw = np.array(y_pred_val_sw)

    eval_sliding_metrics, eval_sliding_report = build_metrics_dict(
        y_true=y_true_val_sliding,
        y_pred=y_pred_val_sw,
        label_list=label_list,
        label2id=label2id,
        split="eval",
        variant="sliding"
    )

    eval_sliding_metrics.update({
        "eval/avg_windows_sliding": float(np.mean(nwin_val)),
        "eval/max_windows_sliding": float(np.max(nwin_val)),
    })

    wandb.log(eval_sliding_metrics)

    print("\nVALIDATION SLIDING")
    print(f"  Accuracy: {eval_sliding_metrics['eval/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {eval_sliding_metrics['eval/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true_val_sliding,
        y_pred_val_sw,
        target_names=label_list,
        zero_division=0
    ))

# =========================================
# 11) TEST simple — resultado final comparable
# =========================================
pred_test_simple = trainer.predict(ds_tok["test"])
logits_test_simple = pred_test_simple.predictions
y_true = pred_test_simple.label_ids
y_pred = np.argmax(logits_test_simple, axis=-1)
probs = softmax_stable(logits_test_simple, axis=1)

true_label_txt = np.array([id2label[i] for i in y_true])
pred_label_txt = np.array([id2label[i] for i in y_pred])

test_meta = test_df.reset_index(drop=True).copy()
test_meta["true_label"] = true_label_txt
test_meta["pred_label"] = pred_label_txt
test_meta["conf_pred"] = probs.max(axis=1)
test_meta["proba_N"] = probs[:, label2id["N"]]
test_meta["proba_Neu"] = probs[:, label2id["Neu"]]
test_meta["proba_P"] = probs[:, label2id["P"]]
test_meta["proba_vector"] = probs.tolist()

test_simple_metrics, test_simple_report = build_metrics_dict(
    y_true=y_true,
    y_pred=y_pred,
    label_list=label_list,
    label2id=label2id,
    split="test",
    variant="simple"
)

wandb.log(test_simple_metrics)

print("\nTEST SIMPLE")
print(f"  Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f}")
print(f"  F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f}")
print(classification_report(
    y_true,
    y_pred,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 12) TEST sliding — resultado final comparable
# =========================================
if USE_SLIDING:
    y_pred_sw = []
    conf_sw = []
    nwin_sw = []
    votes_N_list, votes_Neu_list, votes_P_list = [], [], []

    for txt in test_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_sw.append(label2id[lab])
        conf_sw.append(conf)
        nwin_sw.append(n_win)

        if votes_counts is None:
            votes_N_list.append(np.nan)
            votes_Neu_list.append(np.nan)
            votes_P_list.append(np.nan)
        else:
            votes_N_list.append(int(votes_counts[label2id["N"]]))
            votes_Neu_list.append(int(votes_counts[label2id["Neu"]]))
            votes_P_list.append(int(votes_counts[label2id["P"]]))

    y_pred_sw = np.array(y_pred_sw)

    assert len(test_df) == len(y_true) == len(y_pred_sw), "Mismatch en longitudes de test"

    test_meta["pred_label_sw"] = [id2label[i] for i in y_pred_sw]
    test_meta["conf_sw"] = conf_sw
    test_meta["n_windows"] = nwin_sw
    test_meta["votes_N"] = votes_N_list
    test_meta["votes_Neu"] = votes_Neu_list
    test_meta["votes_P"] = votes_P_list

    test_sliding_metrics, test_sliding_report = build_metrics_dict(
        y_true=y_true,
        y_pred=y_pred_sw,
        label_list=label_list,
        label2id=label2id,
        split="test",
        variant="sliding"
    )

    test_sliding_metrics.update({
        "test/avg_windows_sliding": float(np.mean(nwin_sw)),
        "test/max_windows_sliding": float(np.max(nwin_sw)),
    })

    wandb.log(test_sliding_metrics)

    print("\nTEST SLIDING")
    print(f"  Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true,
        y_pred_sw,
        target_names=label_list,
        zero_division=0
    ))

else:
    test_meta["pred_label_sw"] = np.nan
    test_meta["conf_sw"] = np.nan
    test_meta["n_windows"] = np.nan
    test_meta["votes_N"] = np.nan
    test_meta["votes_Neu"] = np.nan
    test_meta["votes_P"] = np.nan

# =========================================
# 13) Export Excel (test_all + errores simple + errores sliding)
# =========================================
cols_excel = [
    "id_intervencion", "file_name", "n_legislatura", "cuerpo", "fecha",
    "locutor", "encabezado", "intervencion", "n_palabras", "n_caracteres",
    "año", "sexo_locutor", "apellido_locutor_norm", "partido_imputado",
    "sentimiento", "text",
    "true_label",
    "pred_label", "conf_pred", "proba_N", "proba_Neu", "proba_P", "proba_vector",
    "pred_label_sw", "conf_sw", "n_windows", "votes_N", "votes_Neu", "votes_P",
]
cols_excel = [c for c in cols_excel if c in test_meta.columns]

df_errors_simple = test_meta[test_meta["true_label"] != test_meta["pred_label"]].copy()
df_errors_sliding = (
    test_meta[test_meta["true_label"] != test_meta["pred_label_sw"]].copy()
    if USE_SLIDING else pd.DataFrame()
)
df_corrige_sliding = (
    test_meta[
        (test_meta["true_label"] != test_meta["pred_label"]) &
        (test_meta["true_label"] == test_meta["pred_label_sw"])
    ].copy()
    if USE_SLIDING else pd.DataFrame()
)

out_path = f"analisis_{RUN_NAME}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    test_meta[cols_excel].to_excel(writer, index=False, sheet_name="test_all")
    df_errors_simple[cols_excel].to_excel(writer, index=False, sheet_name="errores_simple")
    if USE_SLIDING:
        df_errors_sliding[cols_excel].to_excel(writer, index=False, sheet_name="errores_sliding")
        df_corrige_sliding[cols_excel].to_excel(writer, index=False, sheet_name="corrige_sliding")

print("Excel generado:", out_path)
print(f"  test total:        {len(test_meta)}")
print(f"  errores simple:    {len(df_errors_simple)}")
if USE_SLIDING:
    print(f"  errores sliding:   {len(df_errors_sliding)}")
    print(f"  corrige sliding:   {len(df_corrige_sliding)}")

print("\n=== TEST SIMPLE ===")
print(classification_report(y_true, y_pred, target_names=label_list, zero_division=0))

if USE_SLIDING:
    print("\n=== TEST SLIDING ===")
    print(classification_report(y_true, y_pred_sw, target_names=label_list, zero_division=0))

print("\n" + "=" * 60)
print("RESUMEN FINAL — TEST")
print(f"Simple  | Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f} | "
      f"F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f} | "
      f"F1-weighted: {test_simple_metrics['test/f1_weighted_simple']:.4f}")

if USE_SLIDING:
    print(f"Sliding | Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f} | "
          f"F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f} | "
          f"F1-weighted: {test_sliding_metrics['test/f1_weighted_sliding']:.4f}")
print("=" * 60)

wandb.finish()

RUN: rouberta_base_1405 ID: s2nxz9js USE_SLIDING: False AGG: mean
Train: 979 Val: 122 Test: 123


Map:   0%|          | 0/979 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


VALIDATION SIMPLE
  Accuracy: 0.7869
  F1-macro: 0.7839
              precision    recall  f1-score   support

           N       0.74      0.71      0.72        41
         Neu       0.90      0.84      0.87        45
           P       0.71      0.81      0.75        36

    accuracy                           0.79       122
   macro avg       0.79      0.79      0.78       122
weighted avg       0.79      0.79      0.79       122




TEST SIMPLE
  Accuracy: 0.7154
  F1-macro: 0.7138
              precision    recall  f1-score   support

           N       0.71      0.71      0.71        41
         Neu       0.79      0.72      0.75        46
           P       0.65      0.72      0.68        36

    accuracy                           0.72       123
   macro avg       0.71      0.72      0.71       123
weighted avg       0.72      0.72      0.72       123

Excel generado: analisis_rouberta_base_1405.xlsx
  test total:        123
  errores simple:    35

=== TEST SIMPLE ===
              precision    recall  f1-score   support

           N       0.71      0.71      0.71        41
         Neu       0.79      0.72      0.75        46
           P       0.65      0.72      0.68        36

    accuracy                           0.72       123
   macro avg       0.71      0.72      0.71       123
weighted avg       0.72      0.72      0.72       123


RESUMEN FINAL — TEST
Simple  | Accuracy: 0.7154 | F1-macro: 0.7138 

data/n_test,▁
data/n_train,▁
data/n_val,▁
eval/accuracy,▁▃▇█
eval/accuracy_simple,▁
eval/class_N_accuracy_ovr_simple,▁
eval/class_N_f1_simple,▁
eval/class_N_precision_simple,▁
eval/class_N_recall_simple,▁
eval/class_N_support_simple,▁
+48,...


## Slide + Vote

In [ ]:
# =========================================
# 0) W&B + Config (antes de todo)
# =========================================
import os
import random
import numpy as np
import pandas as pd
import torch
import wandb

WANDB_PROJECT = "Tesis ORT"
WANDB_ENTITY  = "fbenedetti97-universidad-ort-uruguay"

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["WANDB_ENTITY"] = WANDB_ENTITY

if wandb.run is not None:
    wandb.finish()

USE_SLIDING = True
AGG_METHOD  = "vote"   # "mean" / "vote"

# =========================================
# 1) Imports + Config
# =========================================
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

SEED       = 1405
MODEL_CKPT = "pln-udelar/rouberta-base-uy22-cased"
MODEL_TAG  = "rouberta"

RUN_NAME   = (
    f"{MODEL_TAG}_base_{SEED}"
    if not USE_SLIDING
    else f"{MODEL_TAG}_slide_{SEED}_{AGG_METHOD}"
)

MAX_LEN_TRAIN = 128
MAX_LEN_SW    = 128
STRIDE_SW     = 96

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================
# 2) Init W&B
# =========================================
run = wandb.init(
    project=WANDB_PROJECT,
    # entity=WANDB_ENTITY,
    name=RUN_NAME,
    config={
        "seed": int(SEED),
        "model_ckpt": MODEL_CKPT,
        "model_tag": MODEL_TAG,
        "run_type": "finetuned_single_run",
        "max_len_train": int(MAX_LEN_TRAIN),
        "use_sliding": USE_SLIDING,
        "max_len_sw": int(MAX_LEN_SW),
        "stride_sw": int(STRIDE_SW),
        "agg_method": AGG_METHOD,
        "learning_rate": float(2e-5),
        "train_batch_size": int(16),
        "eval_batch_size": int(32),
        "epochs": int(4),
        "weight_decay": float(0.01),
        "selection_metric": "test/f1_macro_sliding" if USE_SLIDING else "test/f1_macro_simple",
    },
    tags=["finetune", "single_run", "rouberta", "sliding" if USE_SLIDING else "simple"],
    reinit="finish_previous",
)

print("RUN:", wandb.run.name, "ID:", wandb.run.id, "USE_SLIDING:", USE_SLIDING, "AGG:", AGG_METHOD)

# =========================================
# 3) Utils
# =========================================
def softmax_stable(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

@torch.no_grad()
def predict_proba_windows(
    text: str,
    model,
    tokenizer,
    max_len: int = 128,
    stride: int = 64,
    device=None
):
    model.eval()
    if device is None:
        device = model.device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    if len(input_ids) > 0 and isinstance(input_ids[0], int):
        input_ids = [input_ids]
        attention_mask = [attention_mask]

    max_batch_len = max(len(x) for x in input_ids)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

    def pad(seq, pad_value):
        return seq + [pad_value] * (max_batch_len - len(seq))

    input_ids = torch.tensor(
        [pad(x, pad_id) for x in input_ids],
        dtype=torch.long,
        device=device
    )
    attention_mask = torch.tensor(
        [pad(x, 0) for x in attention_mask],
        dtype=torch.long,
        device=device
    )

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    return probs, len(probs)

@torch.no_grad()
def predict_text_truncated(text, model, tokenizer, id2label, max_len=128, device=None):
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        return_attention_mask=True,
        padding="max_length",
        return_tensors="pt",
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()[0]

    pred_id = int(np.argmax(probs))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs))
    n_win = 1
    votes_counts = None

    return pred_label, conf, probs, n_win, votes_counts

def aggregate_probs(probs_windows: np.ndarray, method: str = "mean"):
    if method == "mean":
        return probs_windows.mean(axis=0)
    if method == "max":
        return probs_windows.max(axis=0)
    if method == "vote":
        votes = np.argmax(probs_windows, axis=1)
        counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(float)
        return counts / counts.sum()
    raise ValueError("method debe ser 'mean', 'max' o 'vote'")

def predict_text_with_sliding(text, model, tokenizer, id2label,
                              max_len=128, stride=64, agg="mean"):
    """
    Retorna:
      pred_label (str),
      conf (float),
      probs_agg (np.array),
      n_win (int),
      votes_counts (np.array int) o None
    """
    probs_windows, n_win = predict_proba_windows(
        text=text,
        model=model,
        tokenizer=tokenizer,
        max_len=max_len,
        stride=stride
    )
    votes_counts = None

    if agg == "vote":
        votes = np.argmax(probs_windows, axis=1)
        votes_counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(int)
        max_count = votes_counts.max()
        tied = np.where(votes_counts == max_count)[0]

        if len(tied) == 1:
            pred_id = int(tied[0])
        else:
            mean_probs = probs_windows.mean(axis=0)
            pred_id = int(tied[np.argmax(mean_probs[tied])])

        probs_agg = votes_counts.astype(float) / votes_counts.sum()
        conf = float(probs_agg[pred_id])
        pred_label = id2label[pred_id]
        return pred_label, conf, probs_agg, n_win, votes_counts

    probs_agg = aggregate_probs(probs_windows, method=agg)
    pred_id = int(np.argmax(probs_agg))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs_agg))
    return pred_label, conf, probs_agg, n_win, votes_counts

# =========================================
# 4) Datos
# =========================================
df_lab = df_limpio[df_limpio["sentimiento"].notna()].copy()
df_lab["text"] = df_lab["intervencion"].astype(str)

valid_labels = ["N", "Neu", "P"]
df_lab = df_lab[df_lab["sentimiento"].isin(valid_labels)].copy()

label_list = ["N", "Neu", "P"]
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
df_lab["label"] = df_lab["sentimiento"].map(label2id)

# Split 80/10/10
train_df, temp_df = train_test_split(
    df_lab,
    test_size=0.20,
    random_state=SEED,
    stratify=df_lab["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

wandb.log({
    "data/n_train": len(train_df),
    "data/n_val": len(val_df),
    "data/n_test": len(test_df),
})

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# =========================================
# 5) Tokenize
# =========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize_train(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN_TRAIN)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

ds = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})
ds_tok = ds.map(tokenize_train, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================================
# 6) Model
# =========================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# =========================================
# 7) Metrics para Trainer
# =========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# =========================================
# 8) Trainer + Train
# =========================================
use_fp16 = torch.cuda.is_available()
OUTPUT_DIR = f"{MODEL_TAG}_finetuned_{wandb.run.id}"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    fp16=use_fp16,
    report_to="wandb",
    run_name=wandb.run.name,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

best_model = trainer.model

# =========================================
# 9) VALIDATION simple — referencia, no selección final
# =========================================
pred_val_simple = trainer.predict(ds_tok["validation"])
logits_val_simple = pred_val_simple.predictions
y_true_val_simple = pred_val_simple.label_ids
y_pred_val_simple = np.argmax(logits_val_simple, axis=-1)

eval_simple_metrics, eval_simple_report = build_metrics_dict(
    y_true=y_true_val_simple,
    y_pred=y_pred_val_simple,
    label_list=label_list,
    label2id=label2id,
    split="eval",
    variant="simple"
)

wandb.log(eval_simple_metrics)

print("\nVALIDATION SIMPLE")
print(f"  Accuracy: {eval_simple_metrics['eval/accuracy_simple']:.4f}")
print(f"  F1-macro: {eval_simple_metrics['eval/f1_macro_simple']:.4f}")
print(classification_report(
    y_true_val_simple,
    y_pred_val_simple,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 10) VALIDATION sliding — referencia, no selección final
# =========================================
if USE_SLIDING:
    y_true_val_sliding = val_df["label"].to_numpy()
    y_pred_val_sw = []
    nwin_val = []

    for txt in val_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_val_sw.append(label2id[lab])
        nwin_val.append(n_win)

    y_pred_val_sw = np.array(y_pred_val_sw)

    eval_sliding_metrics, eval_sliding_report = build_metrics_dict(
        y_true=y_true_val_sliding,
        y_pred=y_pred_val_sw,
        label_list=label_list,
        label2id=label2id,
        split="eval",
        variant="sliding"
    )

    eval_sliding_metrics.update({
        "eval/avg_windows_sliding": float(np.mean(nwin_val)),
        "eval/max_windows_sliding": float(np.max(nwin_val)),
    })

    wandb.log(eval_sliding_metrics)

    print("\nVALIDATION SLIDING")
    print(f"  Accuracy: {eval_sliding_metrics['eval/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {eval_sliding_metrics['eval/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true_val_sliding,
        y_pred_val_sw,
        target_names=label_list,
        zero_division=0
    ))

# =========================================
# 11) TEST simple — resultado final comparable
# =========================================
pred_test_simple = trainer.predict(ds_tok["test"])
logits_test_simple = pred_test_simple.predictions
y_true = pred_test_simple.label_ids
y_pred = np.argmax(logits_test_simple, axis=-1)
probs = softmax_stable(logits_test_simple, axis=1)

true_label_txt = np.array([id2label[i] for i in y_true])
pred_label_txt = np.array([id2label[i] for i in y_pred])

test_meta = test_df.reset_index(drop=True).copy()
test_meta["true_label"] = true_label_txt
test_meta["pred_label"] = pred_label_txt
test_meta["conf_pred"] = probs.max(axis=1)
test_meta["proba_N"] = probs[:, label2id["N"]]
test_meta["proba_Neu"] = probs[:, label2id["Neu"]]
test_meta["proba_P"] = probs[:, label2id["P"]]
test_meta["proba_vector"] = probs.tolist()

test_simple_metrics, test_simple_report = build_metrics_dict(
    y_true=y_true,
    y_pred=y_pred,
    label_list=label_list,
    label2id=label2id,
    split="test",
    variant="simple"
)

wandb.log(test_simple_metrics)

print("\nTEST SIMPLE")
print(f"  Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f}")
print(f"  F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f}")
print(classification_report(
    y_true,
    y_pred,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 12) TEST sliding — resultado final comparable
# =========================================
if USE_SLIDING:
    y_pred_sw = []
    conf_sw = []
    nwin_sw = []
    votes_N_list, votes_Neu_list, votes_P_list = [], [], []

    for txt in test_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_sw.append(label2id[lab])
        conf_sw.append(conf)
        nwin_sw.append(n_win)

        if votes_counts is None:
            votes_N_list.append(np.nan)
            votes_Neu_list.append(np.nan)
            votes_P_list.append(np.nan)
        else:
            votes_N_list.append(int(votes_counts[label2id["N"]]))
            votes_Neu_list.append(int(votes_counts[label2id["Neu"]]))
            votes_P_list.append(int(votes_counts[label2id["P"]]))

    y_pred_sw = np.array(y_pred_sw)

    assert len(test_df) == len(y_true) == len(y_pred_sw), "Mismatch en longitudes de test"

    test_meta["pred_label_sw"] = [id2label[i] for i in y_pred_sw]
    test_meta["conf_sw"] = conf_sw
    test_meta["n_windows"] = nwin_sw
    test_meta["votes_N"] = votes_N_list
    test_meta["votes_Neu"] = votes_Neu_list
    test_meta["votes_P"] = votes_P_list

    test_sliding_metrics, test_sliding_report = build_metrics_dict(
        y_true=y_true,
        y_pred=y_pred_sw,
        label_list=label_list,
        label2id=label2id,
        split="test",
        variant="sliding"
    )

    test_sliding_metrics.update({
        "test/avg_windows_sliding": float(np.mean(nwin_sw)),
        "test/max_windows_sliding": float(np.max(nwin_sw)),
    })

    wandb.log(test_sliding_metrics)

    print("\nTEST SLIDING")
    print(f"  Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true,
        y_pred_sw,
        target_names=label_list,
        zero_division=0
    ))

else:
    test_meta["pred_label_sw"] = np.nan
    test_meta["conf_sw"] = np.nan
    test_meta["n_windows"] = np.nan
    test_meta["votes_N"] = np.nan
    test_meta["votes_Neu"] = np.nan
    test_meta["votes_P"] = np.nan

# =========================================
# 13) Export Excel (test_all + errores simple + errores sliding)
# =========================================
cols_excel = [
    "id_intervencion", "file_name", "n_legislatura", "cuerpo", "fecha",
    "locutor", "encabezado", "intervencion", "n_palabras", "n_caracteres",
    "año", "sexo_locutor", "apellido_locutor_norm", "partido_imputado",
    "sentimiento", "text",
    "true_label",
    "pred_label", "conf_pred", "proba_N", "proba_Neu", "proba_P", "proba_vector",
    "pred_label_sw", "conf_sw", "n_windows", "votes_N", "votes_Neu", "votes_P",
]
cols_excel = [c for c in cols_excel if c in test_meta.columns]

df_errors_simple = test_meta[test_meta["true_label"] != test_meta["pred_label"]].copy()
df_errors_sliding = (
    test_meta[test_meta["true_label"] != test_meta["pred_label_sw"]].copy()
    if USE_SLIDING else pd.DataFrame()
)
df_corrige_sliding = (
    test_meta[
        (test_meta["true_label"] != test_meta["pred_label"]) &
        (test_meta["true_label"] == test_meta["pred_label_sw"])
    ].copy()
    if USE_SLIDING else pd.DataFrame()
)

out_path = f"analisis_{RUN_NAME}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    test_meta[cols_excel].to_excel(writer, index=False, sheet_name="test_all")
    df_errors_simple[cols_excel].to_excel(writer, index=False, sheet_name="errores_simple")
    if USE_SLIDING:
        df_errors_sliding[cols_excel].to_excel(writer, index=False, sheet_name="errores_sliding")
        df_corrige_sliding[cols_excel].to_excel(writer, index=False, sheet_name="corrige_sliding")

print("Excel generado:", out_path)
print(f"  test total:        {len(test_meta)}")
print(f"  errores simple:    {len(df_errors_simple)}")
if USE_SLIDING:
    print(f"  errores sliding:   {len(df_errors_sliding)}")
    print(f"  corrige sliding:   {len(df_corrige_sliding)}")

print("\n=== TEST SIMPLE ===")
print(classification_report(y_true, y_pred, target_names=label_list, zero_division=0))

if USE_SLIDING:
    print("\n=== TEST SLIDING ===")
    print(classification_report(y_true, y_pred_sw, target_names=label_list, zero_division=0))

print("\n" + "=" * 60)
print("RESUMEN FINAL — TEST")
print(f"Simple  | Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f} | "
      f"F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f} | "
      f"F1-weighted: {test_simple_metrics['test/f1_weighted_simple']:.4f}")

if USE_SLIDING:
    print(f"Sliding | Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f} | "
          f"F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f} | "
          f"F1-weighted: {test_sliding_metrics['test/f1_weighted_sliding']:.4f}")
print("=" * 60)

wandb.finish()

RUN: rouberta_slide_1405_vote ID: csu6ppz9 USE_SLIDING: True AGG: vote
Train: 979 Val: 122 Test: 123


Map:   0%|          | 0/979 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


VALIDATION SIMPLE
  Accuracy: 0.7869
  F1-macro: 0.7839
              precision    recall  f1-score   support

           N       0.74      0.71      0.72        41
         Neu       0.90      0.84      0.87        45
           P       0.71      0.81      0.75        36

    accuracy                           0.79       122
   macro avg       0.79      0.79      0.78       122
weighted avg       0.79      0.79      0.79       122


VALIDATION SLIDING
  Accuracy: 0.7787
  F1-macro: 0.7688
              precision    recall  f1-score   support

           N       0.71      0.88      0.78        41
         Neu       0.83      0.84      0.84        45
           P       0.84      0.58      0.69        36

    accuracy                           0.78       122
   macro avg       0.79      0.77      0.77       122
weighted avg       0.79      0.78      0.77       122




TEST SIMPLE
  Accuracy: 0.7154
  F1-macro: 0.7138
              precision    recall  f1-score   support

           N       0.71      0.71      0.71        41
         Neu       0.79      0.72      0.75        46
           P       0.65      0.72      0.68        36

    accuracy                           0.72       123
   macro avg       0.71      0.72      0.71       123
weighted avg       0.72      0.72      0.72       123


TEST SLIDING
  Accuracy: 0.7236
  F1-macro: 0.7015
              precision    recall  f1-score   support

           N       0.70      0.85      0.77        41
         Neu       0.75      0.83      0.78        46
           P       0.73      0.44      0.55        36

    accuracy                           0.72       123
   macro avg       0.72      0.71      0.70       123
weighted avg       0.72      0.72      0.71       123

Excel generado: analisis_rouberta_slide_1405_vote.xlsx
  test total:        123
  errores simple:    35
  errores sliding:   34
  corri

data/n_test,▁
data/n_train,▁
data/n_val,▁
eval/accuracy,▁▃▇█
eval/accuracy_simple,▁
eval/accuracy_sliding,▁
eval/avg_windows_sliding,▁
eval/class_N_accuracy_ovr_simple,▁
eval/class_N_accuracy_ovr_sliding,▁
eval/class_N_f1_simple,▁
+88,...


## Slide + Mean

In [ ]:
# =========================================
# 0) W&B + Config (antes de todo)
# =========================================
import os
import random
import numpy as np
import pandas as pd
import torch
import wandb

WANDB_PROJECT = "Tesis ORT"
WANDB_ENTITY  = "fbenedetti97-universidad-ort-uruguay"

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["WANDB_ENTITY"] = WANDB_ENTITY

if wandb.run is not None:
    wandb.finish()

USE_SLIDING = True
AGG_METHOD  = "mean"   # "mean" / "vote"

# =========================================
# 1) Imports + Config
# =========================================
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

SEED       = 1405
MODEL_CKPT = "pln-udelar/rouberta-base-uy22-cased"
MODEL_TAG  = "rouberta"

RUN_NAME   = (
    f"{MODEL_TAG}_base_{SEED}"
    if not USE_SLIDING
    else f"{MODEL_TAG}_slide_{SEED}_{AGG_METHOD}"
)

MAX_LEN_TRAIN = 128
MAX_LEN_SW    = 128
STRIDE_SW     = 96

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================
# 2) Init W&B
# =========================================
run = wandb.init(
    project=WANDB_PROJECT,
    # entity=WANDB_ENTITY,
    name=RUN_NAME,
    config={
        "seed": int(SEED),
        "model_ckpt": MODEL_CKPT,
        "model_tag": MODEL_TAG,
        "run_type": "finetuned_single_run",
        "max_len_train": int(MAX_LEN_TRAIN),
        "use_sliding": USE_SLIDING,
        "max_len_sw": int(MAX_LEN_SW),
        "stride_sw": int(STRIDE_SW),
        "agg_method": AGG_METHOD,
        "learning_rate": float(2e-5),
        "train_batch_size": int(16),
        "eval_batch_size": int(32),
        "epochs": int(4),
        "weight_decay": float(0.01),
        "selection_metric": "test/f1_macro_sliding" if USE_SLIDING else "test/f1_macro_simple",
    },
    tags=["finetune", "single_run", "rouberta", "sliding" if USE_SLIDING else "simple"],
    reinit="finish_previous",
)

print("RUN:", wandb.run.name, "ID:", wandb.run.id, "USE_SLIDING:", USE_SLIDING, "AGG:", AGG_METHOD)

# =========================================
# 3) Utils
# =========================================
def softmax_stable(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

@torch.no_grad()
def predict_proba_windows(
    text: str,
    model,
    tokenizer,
    max_len: int = 128,
    stride: int = 64,
    device=None
):
    model.eval()
    if device is None:
        device = model.device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    if len(input_ids) > 0 and isinstance(input_ids[0], int):
        input_ids = [input_ids]
        attention_mask = [attention_mask]

    max_batch_len = max(len(x) for x in input_ids)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

    def pad(seq, pad_value):
        return seq + [pad_value] * (max_batch_len - len(seq))

    input_ids = torch.tensor(
        [pad(x, pad_id) for x in input_ids],
        dtype=torch.long,
        device=device
    )
    attention_mask = torch.tensor(
        [pad(x, 0) for x in attention_mask],
        dtype=torch.long,
        device=device
    )

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    return probs, len(probs)

@torch.no_grad()
def predict_text_truncated(text, model, tokenizer, id2label, max_len=128, device=None):
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        return_attention_mask=True,
        padding="max_length",
        return_tensors="pt",
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()[0]

    pred_id = int(np.argmax(probs))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs))
    n_win = 1
    votes_counts = None

    return pred_label, conf, probs, n_win, votes_counts

def aggregate_probs(probs_windows: np.ndarray, method: str = "mean"):
    if method == "mean":
        return probs_windows.mean(axis=0)
    if method == "max":
        return probs_windows.max(axis=0)
    if method == "vote":
        votes = np.argmax(probs_windows, axis=1)
        counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(float)
        return counts / counts.sum()
    raise ValueError("method debe ser 'mean', 'max' o 'vote'")

def predict_text_with_sliding(text, model, tokenizer, id2label,
                              max_len=128, stride=64, agg="mean"):
    """
    Retorna:
      pred_label (str),
      conf (float),
      probs_agg (np.array),
      n_win (int),
      votes_counts (np.array int) o None
    """
    probs_windows, n_win = predict_proba_windows(
        text=text,
        model=model,
        tokenizer=tokenizer,
        max_len=max_len,
        stride=stride
    )
    votes_counts = None

    if agg == "vote":
        votes = np.argmax(probs_windows, axis=1)
        votes_counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(int)
        max_count = votes_counts.max()
        tied = np.where(votes_counts == max_count)[0]

        if len(tied) == 1:
            pred_id = int(tied[0])
        else:
            mean_probs = probs_windows.mean(axis=0)
            pred_id = int(tied[np.argmax(mean_probs[tied])])

        probs_agg = votes_counts.astype(float) / votes_counts.sum()
        conf = float(probs_agg[pred_id])
        pred_label = id2label[pred_id]
        return pred_label, conf, probs_agg, n_win, votes_counts

    probs_agg = aggregate_probs(probs_windows, method=agg)
    pred_id = int(np.argmax(probs_agg))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs_agg))
    return pred_label, conf, probs_agg, n_win, votes_counts

# =========================================
# 4) Datos
# =========================================
df_lab = df_limpio[df_limpio["sentimiento"].notna()].copy()
df_lab["text"] = df_lab["intervencion"].astype(str)

valid_labels = ["N", "Neu", "P"]
df_lab = df_lab[df_lab["sentimiento"].isin(valid_labels)].copy()

label_list = ["N", "Neu", "P"]
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
df_lab["label"] = df_lab["sentimiento"].map(label2id)

# Split 80/10/10
train_df, temp_df = train_test_split(
    df_lab,
    test_size=0.20,
    random_state=SEED,
    stratify=df_lab["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

wandb.log({
    "data/n_train": len(train_df),
    "data/n_val": len(val_df),
    "data/n_test": len(test_df),
})

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# =========================================
# 5) Tokenize
# =========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize_train(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN_TRAIN)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

ds = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})
ds_tok = ds.map(tokenize_train, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================================
# 6) Model
# =========================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# =========================================
# 7) Metrics para Trainer
# =========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# =========================================
# 8) Trainer + Train
# =========================================
use_fp16 = torch.cuda.is_available()
OUTPUT_DIR = f"{MODEL_TAG}_finetuned_{wandb.run.id}"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    fp16=use_fp16,
    report_to="wandb",
    run_name=wandb.run.name,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

best_model = trainer.model

# =========================================
# 9) VALIDATION simple — referencia, no selección final
# =========================================
pred_val_simple = trainer.predict(ds_tok["validation"])
logits_val_simple = pred_val_simple.predictions
y_true_val_simple = pred_val_simple.label_ids
y_pred_val_simple = np.argmax(logits_val_simple, axis=-1)

eval_simple_metrics, eval_simple_report = build_metrics_dict(
    y_true=y_true_val_simple,
    y_pred=y_pred_val_simple,
    label_list=label_list,
    label2id=label2id,
    split="eval",
    variant="simple"
)

wandb.log(eval_simple_metrics)

print("\nVALIDATION SIMPLE")
print(f"  Accuracy: {eval_simple_metrics['eval/accuracy_simple']:.4f}")
print(f"  F1-macro: {eval_simple_metrics['eval/f1_macro_simple']:.4f}")
print(classification_report(
    y_true_val_simple,
    y_pred_val_simple,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 10) VALIDATION sliding — referencia, no selección final
# =========================================
if USE_SLIDING:
    y_true_val_sliding = val_df["label"].to_numpy()
    y_pred_val_sw = []
    nwin_val = []

    for txt in val_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_val_sw.append(label2id[lab])
        nwin_val.append(n_win)

    y_pred_val_sw = np.array(y_pred_val_sw)

    eval_sliding_metrics, eval_sliding_report = build_metrics_dict(
        y_true=y_true_val_sliding,
        y_pred=y_pred_val_sw,
        label_list=label_list,
        label2id=label2id,
        split="eval",
        variant="sliding"
    )

    eval_sliding_metrics.update({
        "eval/avg_windows_sliding": float(np.mean(nwin_val)),
        "eval/max_windows_sliding": float(np.max(nwin_val)),
    })

    wandb.log(eval_sliding_metrics)

    print("\nVALIDATION SLIDING")
    print(f"  Accuracy: {eval_sliding_metrics['eval/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {eval_sliding_metrics['eval/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true_val_sliding,
        y_pred_val_sw,
        target_names=label_list,
        zero_division=0
    ))

# =========================================
# 11) TEST simple — resultado final comparable
# =========================================
pred_test_simple = trainer.predict(ds_tok["test"])
logits_test_simple = pred_test_simple.predictions
y_true = pred_test_simple.label_ids
y_pred = np.argmax(logits_test_simple, axis=-1)
probs = softmax_stable(logits_test_simple, axis=1)

true_label_txt = np.array([id2label[i] for i in y_true])
pred_label_txt = np.array([id2label[i] for i in y_pred])

test_meta = test_df.reset_index(drop=True).copy()
test_meta["true_label"] = true_label_txt
test_meta["pred_label"] = pred_label_txt
test_meta["conf_pred"] = probs.max(axis=1)
test_meta["proba_N"] = probs[:, label2id["N"]]
test_meta["proba_Neu"] = probs[:, label2id["Neu"]]
test_meta["proba_P"] = probs[:, label2id["P"]]
test_meta["proba_vector"] = probs.tolist()

test_simple_metrics, test_simple_report = build_metrics_dict(
    y_true=y_true,
    y_pred=y_pred,
    label_list=label_list,
    label2id=label2id,
    split="test",
    variant="simple"
)

wandb.log(test_simple_metrics)

print("\nTEST SIMPLE")
print(f"  Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f}")
print(f"  F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f}")
print(classification_report(
    y_true,
    y_pred,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 12) TEST sliding — resultado final comparable
# =========================================
if USE_SLIDING:
    y_pred_sw = []
    conf_sw = []
    nwin_sw = []
    votes_N_list, votes_Neu_list, votes_P_list = [], [], []

    for txt in test_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_sw.append(label2id[lab])
        conf_sw.append(conf)
        nwin_sw.append(n_win)

        if votes_counts is None:
            votes_N_list.append(np.nan)
            votes_Neu_list.append(np.nan)
            votes_P_list.append(np.nan)
        else:
            votes_N_list.append(int(votes_counts[label2id["N"]]))
            votes_Neu_list.append(int(votes_counts[label2id["Neu"]]))
            votes_P_list.append(int(votes_counts[label2id["P"]]))

    y_pred_sw = np.array(y_pred_sw)

    assert len(test_df) == len(y_true) == len(y_pred_sw), "Mismatch en longitudes de test"

    test_meta["pred_label_sw"] = [id2label[i] for i in y_pred_sw]
    test_meta["conf_sw"] = conf_sw
    test_meta["n_windows"] = nwin_sw
    test_meta["votes_N"] = votes_N_list
    test_meta["votes_Neu"] = votes_Neu_list
    test_meta["votes_P"] = votes_P_list

    test_sliding_metrics, test_sliding_report = build_metrics_dict(
        y_true=y_true,
        y_pred=y_pred_sw,
        label_list=label_list,
        label2id=label2id,
        split="test",
        variant="sliding"
    )

    test_sliding_metrics.update({
        "test/avg_windows_sliding": float(np.mean(nwin_sw)),
        "test/max_windows_sliding": float(np.max(nwin_sw)),
    })

    wandb.log(test_sliding_metrics)

    print("\nTEST SLIDING")
    print(f"  Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true,
        y_pred_sw,
        target_names=label_list,
        zero_division=0
    ))

else:
    test_meta["pred_label_sw"] = np.nan
    test_meta["conf_sw"] = np.nan
    test_meta["n_windows"] = np.nan
    test_meta["votes_N"] = np.nan
    test_meta["votes_Neu"] = np.nan
    test_meta["votes_P"] = np.nan

# =========================================
# 13) Export Excel (test_all + errores simple + errores sliding)
# =========================================
cols_excel = [
    "id_intervencion", "file_name", "n_legislatura", "cuerpo", "fecha",
    "locutor", "encabezado", "intervencion", "n_palabras", "n_caracteres",
    "año", "sexo_locutor", "apellido_locutor_norm", "partido_imputado",
    "sentimiento", "text",
    "true_label",
    "pred_label", "conf_pred", "proba_N", "proba_Neu", "proba_P", "proba_vector",
    "pred_label_sw", "conf_sw", "n_windows", "votes_N", "votes_Neu", "votes_P",
]
cols_excel = [c for c in cols_excel if c in test_meta.columns]

df_errors_simple = test_meta[test_meta["true_label"] != test_meta["pred_label"]].copy()
df_errors_sliding = (
    test_meta[test_meta["true_label"] != test_meta["pred_label_sw"]].copy()
    if USE_SLIDING else pd.DataFrame()
)
df_corrige_sliding = (
    test_meta[
        (test_meta["true_label"] != test_meta["pred_label"]) &
        (test_meta["true_label"] == test_meta["pred_label_sw"])
    ].copy()
    if USE_SLIDING else pd.DataFrame()
)

out_path = f"analisis_{RUN_NAME}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    test_meta[cols_excel].to_excel(writer, index=False, sheet_name="test_all")
    df_errors_simple[cols_excel].to_excel(writer, index=False, sheet_name="errores_simple")
    if USE_SLIDING:
        df_errors_sliding[cols_excel].to_excel(writer, index=False, sheet_name="errores_sliding")
        df_corrige_sliding[cols_excel].to_excel(writer, index=False, sheet_name="corrige_sliding")

print("Excel generado:", out_path)
print(f"  test total:        {len(test_meta)}")
print(f"  errores simple:    {len(df_errors_simple)}")
if USE_SLIDING:
    print(f"  errores sliding:   {len(df_errors_sliding)}")
    print(f"  corrige sliding:   {len(df_corrige_sliding)}")

print("\n=== TEST SIMPLE ===")
print(classification_report(y_true, y_pred, target_names=label_list, zero_division=0))

if USE_SLIDING:
    print("\n=== TEST SLIDING ===")
    print(classification_report(y_true, y_pred_sw, target_names=label_list, zero_division=0))

print("\n" + "=" * 60)
print("RESUMEN FINAL — TEST")
print(f"Simple  | Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f} | "
      f"F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f} | "
      f"F1-weighted: {test_simple_metrics['test/f1_weighted_simple']:.4f}")

if USE_SLIDING:
    print(f"Sliding | Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f} | "
          f"F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f} | "
          f"F1-weighted: {test_sliding_metrics['test/f1_weighted_sliding']:.4f}")
print("=" * 60)

wandb.finish()

RUN: rouberta_slide_1405_mean ID: lcyyrqry USE_SLIDING: True AGG: mean
Train: 979 Val: 122 Test: 123


Map:   0%|          | 0/979 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


VALIDATION SIMPLE
  Accuracy: 0.7869
  F1-macro: 0.7839
              precision    recall  f1-score   support

           N       0.74      0.71      0.72        41
         Neu       0.90      0.84      0.87        45
           P       0.71      0.81      0.75        36

    accuracy                           0.79       122
   macro avg       0.79      0.79      0.78       122
weighted avg       0.79      0.79      0.79       122


VALIDATION SLIDING
  Accuracy: 0.7869
  F1-macro: 0.7756
              precision    recall  f1-score   support

           N       0.71      0.88      0.78        41
         Neu       0.87      0.87      0.87        45
           P       0.81      0.58      0.68        36

    accuracy                           0.79       122
   macro avg       0.79      0.78      0.78       122
weighted avg       0.80      0.79      0.78       122




TEST SIMPLE
  Accuracy: 0.7154
  F1-macro: 0.7138
              precision    recall  f1-score   support

           N       0.71      0.71      0.71        41
         Neu       0.79      0.72      0.75        46
           P       0.65      0.72      0.68        36

    accuracy                           0.72       123
   macro avg       0.71      0.72      0.71       123
weighted avg       0.72      0.72      0.72       123


TEST SLIDING
  Accuracy: 0.7154
  F1-macro: 0.6945
              precision    recall  f1-score   support

           N       0.67      0.85      0.75        41
         Neu       0.76      0.80      0.78        46
           P       0.73      0.44      0.55        36

    accuracy                           0.72       123
   macro avg       0.72      0.70      0.69       123
weighted avg       0.72      0.72      0.70       123

Excel generado: analisis_rouberta_slide_1405_mean.xlsx
  test total:        123
  errores simple:    35
  errores sliding:   35
  corri

data/n_test,▁
data/n_train,▁
data/n_val,▁
eval/accuracy,▁▃▇█
eval/accuracy_simple,▁
eval/accuracy_sliding,▁
eval/avg_windows_sliding,▁
eval/class_N_accuracy_ovr_simple,▁
eval/class_N_accuracy_ovr_sliding,▁
eval/class_N_f1_simple,▁
+88,...


# Con SEED 2050



## Truncado

In [ ]:
# =========================================
# 0) W&B + Config (antes de todo)
# =========================================
import os
import random
import numpy as np
import pandas as pd
import torch
import wandb

WANDB_PROJECT = "Tesis ORT"
WANDB_ENTITY  = "fbenedetti97-universidad-ort-uruguay"

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["WANDB_ENTITY"] = WANDB_ENTITY

if wandb.run is not None:
    wandb.finish()

USE_SLIDING = False
AGG_METHOD  = "mean"   # "mean" / "vote"

# =========================================
# 1) Imports + Config
# =========================================
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

SEED       = 2050
MODEL_CKPT = "pln-udelar/rouberta-base-uy22-cased"
MODEL_TAG  = "rouberta"

RUN_NAME   = (
    f"{MODEL_TAG}_base_{SEED}"
    if not USE_SLIDING
    else f"{MODEL_TAG}_slide_{SEED}_{AGG_METHOD}"
)

MAX_LEN_TRAIN = 128
MAX_LEN_SW    = 128
STRIDE_SW     = 96

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================
# 2) Init W&B
# =========================================
run = wandb.init(
    project=WANDB_PROJECT,
    # entity=WANDB_ENTITY,
    name=RUN_NAME,
    config={
        "seed": int(SEED),
        "model_ckpt": MODEL_CKPT,
        "model_tag": MODEL_TAG,
        "run_type": "finetuned_single_run",
        "max_len_train": int(MAX_LEN_TRAIN),
        "use_sliding": USE_SLIDING,
        "max_len_sw": int(MAX_LEN_SW),
        "stride_sw": int(STRIDE_SW),
        "agg_method": AGG_METHOD,
        "learning_rate": float(2e-5),
        "train_batch_size": int(16),
        "eval_batch_size": int(32),
        "epochs": int(4),
        "weight_decay": float(0.01),
        "selection_metric": "test/f1_macro_sliding" if USE_SLIDING else "test/f1_macro_simple",
    },
    tags=["finetune", "single_run", "rouberta", "sliding" if USE_SLIDING else "simple"],
    reinit="finish_previous",
)

print("RUN:", wandb.run.name, "ID:", wandb.run.id, "USE_SLIDING:", USE_SLIDING, "AGG:", AGG_METHOD)

# =========================================
# 3) Utils
# =========================================
def softmax_stable(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

@torch.no_grad()
def predict_proba_windows(
    text: str,
    model,
    tokenizer,
    max_len: int = 128,
    stride: int = 64,
    device=None
):
    model.eval()
    if device is None:
        device = model.device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    if len(input_ids) > 0 and isinstance(input_ids[0], int):
        input_ids = [input_ids]
        attention_mask = [attention_mask]

    max_batch_len = max(len(x) for x in input_ids)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

    def pad(seq, pad_value):
        return seq + [pad_value] * (max_batch_len - len(seq))

    input_ids = torch.tensor(
        [pad(x, pad_id) for x in input_ids],
        dtype=torch.long,
        device=device
    )
    attention_mask = torch.tensor(
        [pad(x, 0) for x in attention_mask],
        dtype=torch.long,
        device=device
    )

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    return probs, len(probs)

@torch.no_grad()
def predict_text_truncated(text, model, tokenizer, id2label, max_len=128, device=None):
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        return_attention_mask=True,
        padding="max_length",
        return_tensors="pt",
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()[0]

    pred_id = int(np.argmax(probs))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs))
    n_win = 1
    votes_counts = None

    return pred_label, conf, probs, n_win, votes_counts

def aggregate_probs(probs_windows: np.ndarray, method: str = "mean"):
    if method == "mean":
        return probs_windows.mean(axis=0)
    if method == "max":
        return probs_windows.max(axis=0)
    if method == "vote":
        votes = np.argmax(probs_windows, axis=1)
        counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(float)
        return counts / counts.sum()
    raise ValueError("method debe ser 'mean', 'max' o 'vote'")

def predict_text_with_sliding(text, model, tokenizer, id2label,
                              max_len=128, stride=64, agg="mean"):
    """
    Retorna:
      pred_label (str),
      conf (float),
      probs_agg (np.array),
      n_win (int),
      votes_counts (np.array int) o None
    """
    probs_windows, n_win = predict_proba_windows(
        text=text,
        model=model,
        tokenizer=tokenizer,
        max_len=max_len,
        stride=stride
    )
    votes_counts = None

    if agg == "vote":
        votes = np.argmax(probs_windows, axis=1)
        votes_counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(int)
        max_count = votes_counts.max()
        tied = np.where(votes_counts == max_count)[0]

        if len(tied) == 1:
            pred_id = int(tied[0])
        else:
            mean_probs = probs_windows.mean(axis=0)
            pred_id = int(tied[np.argmax(mean_probs[tied])])

        probs_agg = votes_counts.astype(float) / votes_counts.sum()
        conf = float(probs_agg[pred_id])
        pred_label = id2label[pred_id]
        return pred_label, conf, probs_agg, n_win, votes_counts

    probs_agg = aggregate_probs(probs_windows, method=agg)
    pred_id = int(np.argmax(probs_agg))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs_agg))
    return pred_label, conf, probs_agg, n_win, votes_counts

# =========================================
# 4) Datos
# =========================================
df_lab = df_limpio[df_limpio["sentimiento"].notna()].copy()
df_lab["text"] = df_lab["intervencion"].astype(str)

valid_labels = ["N", "Neu", "P"]
df_lab = df_lab[df_lab["sentimiento"].isin(valid_labels)].copy()

label_list = ["N", "Neu", "P"]
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
df_lab["label"] = df_lab["sentimiento"].map(label2id)

# Split 80/10/10
train_df, temp_df = train_test_split(
    df_lab,
    test_size=0.20,
    random_state=SEED,
    stratify=df_lab["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

wandb.log({
    "data/n_train": len(train_df),
    "data/n_val": len(val_df),
    "data/n_test": len(test_df),
})

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# =========================================
# 5) Tokenize
# =========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize_train(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN_TRAIN)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

ds = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})
ds_tok = ds.map(tokenize_train, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================================
# 6) Model
# =========================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# =========================================
# 7) Metrics para Trainer
# =========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# =========================================
# 8) Trainer + Train
# =========================================
use_fp16 = torch.cuda.is_available()
OUTPUT_DIR = f"{MODEL_TAG}_finetuned_{wandb.run.id}"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    fp16=use_fp16,
    report_to="wandb",
    run_name=wandb.run.name,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

best_model = trainer.model

# =========================================
# 9) VALIDATION simple — referencia, no selección final
# =========================================
pred_val_simple = trainer.predict(ds_tok["validation"])
logits_val_simple = pred_val_simple.predictions
y_true_val_simple = pred_val_simple.label_ids
y_pred_val_simple = np.argmax(logits_val_simple, axis=-1)

eval_simple_metrics, eval_simple_report = build_metrics_dict(
    y_true=y_true_val_simple,
    y_pred=y_pred_val_simple,
    label_list=label_list,
    label2id=label2id,
    split="eval",
    variant="simple"
)

wandb.log(eval_simple_metrics)

print("\nVALIDATION SIMPLE")
print(f"  Accuracy: {eval_simple_metrics['eval/accuracy_simple']:.4f}")
print(f"  F1-macro: {eval_simple_metrics['eval/f1_macro_simple']:.4f}")
print(classification_report(
    y_true_val_simple,
    y_pred_val_simple,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 10) VALIDATION sliding — referencia, no selección final
# =========================================
if USE_SLIDING:
    y_true_val_sliding = val_df["label"].to_numpy()
    y_pred_val_sw = []
    nwin_val = []

    for txt in val_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_val_sw.append(label2id[lab])
        nwin_val.append(n_win)

    y_pred_val_sw = np.array(y_pred_val_sw)

    eval_sliding_metrics, eval_sliding_report = build_metrics_dict(
        y_true=y_true_val_sliding,
        y_pred=y_pred_val_sw,
        label_list=label_list,
        label2id=label2id,
        split="eval",
        variant="sliding"
    )

    eval_sliding_metrics.update({
        "eval/avg_windows_sliding": float(np.mean(nwin_val)),
        "eval/max_windows_sliding": float(np.max(nwin_val)),
    })

    wandb.log(eval_sliding_metrics)

    print("\nVALIDATION SLIDING")
    print(f"  Accuracy: {eval_sliding_metrics['eval/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {eval_sliding_metrics['eval/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true_val_sliding,
        y_pred_val_sw,
        target_names=label_list,
        zero_division=0
    ))

# =========================================
# 11) TEST simple — resultado final comparable
# =========================================
pred_test_simple = trainer.predict(ds_tok["test"])
logits_test_simple = pred_test_simple.predictions
y_true = pred_test_simple.label_ids
y_pred = np.argmax(logits_test_simple, axis=-1)
probs = softmax_stable(logits_test_simple, axis=1)

true_label_txt = np.array([id2label[i] for i in y_true])
pred_label_txt = np.array([id2label[i] for i in y_pred])

test_meta = test_df.reset_index(drop=True).copy()
test_meta["true_label"] = true_label_txt
test_meta["pred_label"] = pred_label_txt
test_meta["conf_pred"] = probs.max(axis=1)
test_meta["proba_N"] = probs[:, label2id["N"]]
test_meta["proba_Neu"] = probs[:, label2id["Neu"]]
test_meta["proba_P"] = probs[:, label2id["P"]]
test_meta["proba_vector"] = probs.tolist()

test_simple_metrics, test_simple_report = build_metrics_dict(
    y_true=y_true,
    y_pred=y_pred,
    label_list=label_list,
    label2id=label2id,
    split="test",
    variant="simple"
)

wandb.log(test_simple_metrics)

print("\nTEST SIMPLE")
print(f"  Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f}")
print(f"  F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f}")
print(classification_report(
    y_true,
    y_pred,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 12) TEST sliding — resultado final comparable
# =========================================
if USE_SLIDING:
    y_pred_sw = []
    conf_sw = []
    nwin_sw = []
    votes_N_list, votes_Neu_list, votes_P_list = [], [], []

    for txt in test_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_sw.append(label2id[lab])
        conf_sw.append(conf)
        nwin_sw.append(n_win)

        if votes_counts is None:
            votes_N_list.append(np.nan)
            votes_Neu_list.append(np.nan)
            votes_P_list.append(np.nan)
        else:
            votes_N_list.append(int(votes_counts[label2id["N"]]))
            votes_Neu_list.append(int(votes_counts[label2id["Neu"]]))
            votes_P_list.append(int(votes_counts[label2id["P"]]))

    y_pred_sw = np.array(y_pred_sw)

    assert len(test_df) == len(y_true) == len(y_pred_sw), "Mismatch en longitudes de test"

    test_meta["pred_label_sw"] = [id2label[i] for i in y_pred_sw]
    test_meta["conf_sw"] = conf_sw
    test_meta["n_windows"] = nwin_sw
    test_meta["votes_N"] = votes_N_list
    test_meta["votes_Neu"] = votes_Neu_list
    test_meta["votes_P"] = votes_P_list

    test_sliding_metrics, test_sliding_report = build_metrics_dict(
        y_true=y_true,
        y_pred=y_pred_sw,
        label_list=label_list,
        label2id=label2id,
        split="test",
        variant="sliding"
    )

    test_sliding_metrics.update({
        "test/avg_windows_sliding": float(np.mean(nwin_sw)),
        "test/max_windows_sliding": float(np.max(nwin_sw)),
    })

    wandb.log(test_sliding_metrics)

    print("\nTEST SLIDING")
    print(f"  Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true,
        y_pred_sw,
        target_names=label_list,
        zero_division=0
    ))

else:
    test_meta["pred_label_sw"] = np.nan
    test_meta["conf_sw"] = np.nan
    test_meta["n_windows"] = np.nan
    test_meta["votes_N"] = np.nan
    test_meta["votes_Neu"] = np.nan
    test_meta["votes_P"] = np.nan

# =========================================
# 13) Export Excel (test_all + errores simple + errores sliding)
# =========================================
cols_excel = [
    "id_intervencion", "file_name", "n_legislatura", "cuerpo", "fecha",
    "locutor", "encabezado", "intervencion", "n_palabras", "n_caracteres",
    "año", "sexo_locutor", "apellido_locutor_norm", "partido_imputado",
    "sentimiento", "text",
    "true_label",
    "pred_label", "conf_pred", "proba_N", "proba_Neu", "proba_P", "proba_vector",
    "pred_label_sw", "conf_sw", "n_windows", "votes_N", "votes_Neu", "votes_P",
]
cols_excel = [c for c in cols_excel if c in test_meta.columns]

df_errors_simple = test_meta[test_meta["true_label"] != test_meta["pred_label"]].copy()
df_errors_sliding = (
    test_meta[test_meta["true_label"] != test_meta["pred_label_sw"]].copy()
    if USE_SLIDING else pd.DataFrame()
)
df_corrige_sliding = (
    test_meta[
        (test_meta["true_label"] != test_meta["pred_label"]) &
        (test_meta["true_label"] == test_meta["pred_label_sw"])
    ].copy()
    if USE_SLIDING else pd.DataFrame()
)

out_path = f"analisis_{RUN_NAME}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    test_meta[cols_excel].to_excel(writer, index=False, sheet_name="test_all")
    df_errors_simple[cols_excel].to_excel(writer, index=False, sheet_name="errores_simple")
    if USE_SLIDING:
        df_errors_sliding[cols_excel].to_excel(writer, index=False, sheet_name="errores_sliding")
        df_corrige_sliding[cols_excel].to_excel(writer, index=False, sheet_name="corrige_sliding")

print("Excel generado:", out_path)
print(f"  test total:        {len(test_meta)}")
print(f"  errores simple:    {len(df_errors_simple)}")
if USE_SLIDING:
    print(f"  errores sliding:   {len(df_errors_sliding)}")
    print(f"  corrige sliding:   {len(df_corrige_sliding)}")

print("\n=== TEST SIMPLE ===")
print(classification_report(y_true, y_pred, target_names=label_list, zero_division=0))

if USE_SLIDING:
    print("\n=== TEST SLIDING ===")
    print(classification_report(y_true, y_pred_sw, target_names=label_list, zero_division=0))

print("\n" + "=" * 60)
print("RESUMEN FINAL — TEST")
print(f"Simple  | Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f} | "
      f"F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f} | "
      f"F1-weighted: {test_simple_metrics['test/f1_weighted_simple']:.4f}")

if USE_SLIDING:
    print(f"Sliding | Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f} | "
          f"F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f} | "
          f"F1-weighted: {test_sliding_metrics['test/f1_weighted_sliding']:.4f}")
print("=" * 60)

wandb.finish()

RUN: rouberta_base_2050 ID: ux6hu75l USE_SLIDING: False AGG: mean
Train: 979 Val: 122 Test: 123


Map:   0%|          | 0/979 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


VALIDATION SIMPLE
  Accuracy: 0.6721
  F1-macro: 0.6642
              precision    recall  f1-score   support

           N       0.69      0.61      0.65        41
         Neu       0.68      0.80      0.73        45
           P       0.64      0.58      0.61        36

    accuracy                           0.67       122
   macro avg       0.67      0.66      0.66       122
weighted avg       0.67      0.67      0.67       122




TEST SIMPLE
  Accuracy: 0.8130
  F1-macro: 0.8118
              precision    recall  f1-score   support

           N       0.74      0.83      0.78        41
         Neu       0.85      0.85      0.85        46
           P       0.87      0.75      0.81        36

    accuracy                           0.81       123
   macro avg       0.82      0.81      0.81       123
weighted avg       0.82      0.81      0.81       123

Excel generado: analisis_rouberta_base_2050.xlsx
  test total:        123
  errores simple:    23

=== TEST SIMPLE ===
              precision    recall  f1-score   support

           N       0.74      0.83      0.78        41
         Neu       0.85      0.85      0.85        46
           P       0.87      0.75      0.81        36

    accuracy                           0.81       123
   macro avg       0.82      0.81      0.81       123
weighted avg       0.82      0.81      0.81       123


RESUMEN FINAL — TEST
Simple  | Accuracy: 0.8130 | F1-macro: 0.8118 

data/n_test,▁
data/n_train,▁
data/n_val,▁
eval/accuracy,▁███
eval/accuracy_simple,▁
eval/class_N_accuracy_ovr_simple,▁
eval/class_N_f1_simple,▁
eval/class_N_precision_simple,▁
eval/class_N_recall_simple,▁
eval/class_N_support_simple,▁
+48,...


## Slide + Vote

In [ ]:
# =========================================
# 0) W&B + Config (antes de todo)
# =========================================
import os
import random
import numpy as np
import pandas as pd
import torch
import wandb

WANDB_PROJECT = "Tesis ORT"
WANDB_ENTITY  = "fbenedetti97-universidad-ort-uruguay"

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["WANDB_ENTITY"] = WANDB_ENTITY

if wandb.run is not None:
    wandb.finish()

USE_SLIDING = True
AGG_METHOD  = "vote"   # "mean" / "vote"

# =========================================
# 1) Imports + Config
# =========================================
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

SEED       = 2050
MODEL_CKPT = "pln-udelar/rouberta-base-uy22-cased"
MODEL_TAG  = "rouberta"

RUN_NAME   = (
    f"{MODEL_TAG}_base_{SEED}"
    if not USE_SLIDING
    else f"{MODEL_TAG}_slide_{SEED}_{AGG_METHOD}"
)

MAX_LEN_TRAIN = 128
MAX_LEN_SW    = 128
STRIDE_SW     = 96

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================
# 2) Init W&B
# =========================================
run = wandb.init(
    project=WANDB_PROJECT,
    # entity=WANDB_ENTITY,
    name=RUN_NAME,
    config={
        "seed": int(SEED),
        "model_ckpt": MODEL_CKPT,
        "model_tag": MODEL_TAG,
        "run_type": "finetuned_single_run",
        "max_len_train": int(MAX_LEN_TRAIN),
        "use_sliding": USE_SLIDING,
        "max_len_sw": int(MAX_LEN_SW),
        "stride_sw": int(STRIDE_SW),
        "agg_method": AGG_METHOD,
        "learning_rate": float(2e-5),
        "train_batch_size": int(16),
        "eval_batch_size": int(32),
        "epochs": int(4),
        "weight_decay": float(0.01),
        "selection_metric": "test/f1_macro_sliding" if USE_SLIDING else "test/f1_macro_simple",
    },
    tags=["finetune", "single_run", "rouberta", "sliding" if USE_SLIDING else "simple"],
    reinit="finish_previous",
)

print("RUN:", wandb.run.name, "ID:", wandb.run.id, "USE_SLIDING:", USE_SLIDING, "AGG:", AGG_METHOD)

# =========================================
# 3) Utils
# =========================================
def softmax_stable(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

@torch.no_grad()
def predict_proba_windows(
    text: str,
    model,
    tokenizer,
    max_len: int = 128,
    stride: int = 64,
    device=None
):
    model.eval()
    if device is None:
        device = model.device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    if len(input_ids) > 0 and isinstance(input_ids[0], int):
        input_ids = [input_ids]
        attention_mask = [attention_mask]

    max_batch_len = max(len(x) for x in input_ids)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

    def pad(seq, pad_value):
        return seq + [pad_value] * (max_batch_len - len(seq))

    input_ids = torch.tensor(
        [pad(x, pad_id) for x in input_ids],
        dtype=torch.long,
        device=device
    )
    attention_mask = torch.tensor(
        [pad(x, 0) for x in attention_mask],
        dtype=torch.long,
        device=device
    )

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    return probs, len(probs)

@torch.no_grad()
def predict_text_truncated(text, model, tokenizer, id2label, max_len=128, device=None):
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        return_attention_mask=True,
        padding="max_length",
        return_tensors="pt",
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()[0]

    pred_id = int(np.argmax(probs))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs))
    n_win = 1
    votes_counts = None

    return pred_label, conf, probs, n_win, votes_counts

def aggregate_probs(probs_windows: np.ndarray, method: str = "mean"):
    if method == "mean":
        return probs_windows.mean(axis=0)
    if method == "max":
        return probs_windows.max(axis=0)
    if method == "vote":
        votes = np.argmax(probs_windows, axis=1)
        counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(float)
        return counts / counts.sum()
    raise ValueError("method debe ser 'mean', 'max' o 'vote'")

def predict_text_with_sliding(text, model, tokenizer, id2label,
                              max_len=128, stride=64, agg="mean"):
    """
    Retorna:
      pred_label (str),
      conf (float),
      probs_agg (np.array),
      n_win (int),
      votes_counts (np.array int) o None
    """
    probs_windows, n_win = predict_proba_windows(
        text=text,
        model=model,
        tokenizer=tokenizer,
        max_len=max_len,
        stride=stride
    )
    votes_counts = None

    if agg == "vote":
        votes = np.argmax(probs_windows, axis=1)
        votes_counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(int)
        max_count = votes_counts.max()
        tied = np.where(votes_counts == max_count)[0]

        if len(tied) == 1:
            pred_id = int(tied[0])
        else:
            mean_probs = probs_windows.mean(axis=0)
            pred_id = int(tied[np.argmax(mean_probs[tied])])

        probs_agg = votes_counts.astype(float) / votes_counts.sum()
        conf = float(probs_agg[pred_id])
        pred_label = id2label[pred_id]
        return pred_label, conf, probs_agg, n_win, votes_counts

    probs_agg = aggregate_probs(probs_windows, method=agg)
    pred_id = int(np.argmax(probs_agg))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs_agg))
    return pred_label, conf, probs_agg, n_win, votes_counts

# =========================================
# 4) Datos
# =========================================
df_lab = df_limpio[df_limpio["sentimiento"].notna()].copy()
df_lab["text"] = df_lab["intervencion"].astype(str)

valid_labels = ["N", "Neu", "P"]
df_lab = df_lab[df_lab["sentimiento"].isin(valid_labels)].copy()

label_list = ["N", "Neu", "P"]
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
df_lab["label"] = df_lab["sentimiento"].map(label2id)

# Split 80/10/10
train_df, temp_df = train_test_split(
    df_lab,
    test_size=0.20,
    random_state=SEED,
    stratify=df_lab["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

wandb.log({
    "data/n_train": len(train_df),
    "data/n_val": len(val_df),
    "data/n_test": len(test_df),
})

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# =========================================
# 5) Tokenize
# =========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize_train(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN_TRAIN)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

ds = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})
ds_tok = ds.map(tokenize_train, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================================
# 6) Model
# =========================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# =========================================
# 7) Metrics para Trainer
# =========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# =========================================
# 8) Trainer + Train
# =========================================
use_fp16 = torch.cuda.is_available()
OUTPUT_DIR = f"{MODEL_TAG}_finetuned_{wandb.run.id}"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    fp16=use_fp16,
    report_to="wandb",
    run_name=wandb.run.name,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

best_model = trainer.model

# =========================================
# 9) VALIDATION simple — referencia, no selección final
# =========================================
pred_val_simple = trainer.predict(ds_tok["validation"])
logits_val_simple = pred_val_simple.predictions
y_true_val_simple = pred_val_simple.label_ids
y_pred_val_simple = np.argmax(logits_val_simple, axis=-1)

eval_simple_metrics, eval_simple_report = build_metrics_dict(
    y_true=y_true_val_simple,
    y_pred=y_pred_val_simple,
    label_list=label_list,
    label2id=label2id,
    split="eval",
    variant="simple"
)

wandb.log(eval_simple_metrics)

print("\nVALIDATION SIMPLE")
print(f"  Accuracy: {eval_simple_metrics['eval/accuracy_simple']:.4f}")
print(f"  F1-macro: {eval_simple_metrics['eval/f1_macro_simple']:.4f}")
print(classification_report(
    y_true_val_simple,
    y_pred_val_simple,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 10) VALIDATION sliding — referencia, no selección final
# =========================================
if USE_SLIDING:
    y_true_val_sliding = val_df["label"].to_numpy()
    y_pred_val_sw = []
    nwin_val = []

    for txt in val_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_val_sw.append(label2id[lab])
        nwin_val.append(n_win)

    y_pred_val_sw = np.array(y_pred_val_sw)

    eval_sliding_metrics, eval_sliding_report = build_metrics_dict(
        y_true=y_true_val_sliding,
        y_pred=y_pred_val_sw,
        label_list=label_list,
        label2id=label2id,
        split="eval",
        variant="sliding"
    )

    eval_sliding_metrics.update({
        "eval/avg_windows_sliding": float(np.mean(nwin_val)),
        "eval/max_windows_sliding": float(np.max(nwin_val)),
    })

    wandb.log(eval_sliding_metrics)

    print("\nVALIDATION SLIDING")
    print(f"  Accuracy: {eval_sliding_metrics['eval/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {eval_sliding_metrics['eval/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true_val_sliding,
        y_pred_val_sw,
        target_names=label_list,
        zero_division=0
    ))

# =========================================
# 11) TEST simple — resultado final comparable
# =========================================
pred_test_simple = trainer.predict(ds_tok["test"])
logits_test_simple = pred_test_simple.predictions
y_true = pred_test_simple.label_ids
y_pred = np.argmax(logits_test_simple, axis=-1)
probs = softmax_stable(logits_test_simple, axis=1)

true_label_txt = np.array([id2label[i] for i in y_true])
pred_label_txt = np.array([id2label[i] for i in y_pred])

test_meta = test_df.reset_index(drop=True).copy()
test_meta["true_label"] = true_label_txt
test_meta["pred_label"] = pred_label_txt
test_meta["conf_pred"] = probs.max(axis=1)
test_meta["proba_N"] = probs[:, label2id["N"]]
test_meta["proba_Neu"] = probs[:, label2id["Neu"]]
test_meta["proba_P"] = probs[:, label2id["P"]]
test_meta["proba_vector"] = probs.tolist()

test_simple_metrics, test_simple_report = build_metrics_dict(
    y_true=y_true,
    y_pred=y_pred,
    label_list=label_list,
    label2id=label2id,
    split="test",
    variant="simple"
)

wandb.log(test_simple_metrics)

print("\nTEST SIMPLE")
print(f"  Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f}")
print(f"  F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f}")
print(classification_report(
    y_true,
    y_pred,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 12) TEST sliding — resultado final comparable
# =========================================
if USE_SLIDING:
    y_pred_sw = []
    conf_sw = []
    nwin_sw = []
    votes_N_list, votes_Neu_list, votes_P_list = [], [], []

    for txt in test_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_sw.append(label2id[lab])
        conf_sw.append(conf)
        nwin_sw.append(n_win)

        if votes_counts is None:
            votes_N_list.append(np.nan)
            votes_Neu_list.append(np.nan)
            votes_P_list.append(np.nan)
        else:
            votes_N_list.append(int(votes_counts[label2id["N"]]))
            votes_Neu_list.append(int(votes_counts[label2id["Neu"]]))
            votes_P_list.append(int(votes_counts[label2id["P"]]))

    y_pred_sw = np.array(y_pred_sw)

    assert len(test_df) == len(y_true) == len(y_pred_sw), "Mismatch en longitudes de test"

    test_meta["pred_label_sw"] = [id2label[i] for i in y_pred_sw]
    test_meta["conf_sw"] = conf_sw
    test_meta["n_windows"] = nwin_sw
    test_meta["votes_N"] = votes_N_list
    test_meta["votes_Neu"] = votes_Neu_list
    test_meta["votes_P"] = votes_P_list

    test_sliding_metrics, test_sliding_report = build_metrics_dict(
        y_true=y_true,
        y_pred=y_pred_sw,
        label_list=label_list,
        label2id=label2id,
        split="test",
        variant="sliding"
    )

    test_sliding_metrics.update({
        "test/avg_windows_sliding": float(np.mean(nwin_sw)),
        "test/max_windows_sliding": float(np.max(nwin_sw)),
    })

    wandb.log(test_sliding_metrics)

    print("\nTEST SLIDING")
    print(f"  Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true,
        y_pred_sw,
        target_names=label_list,
        zero_division=0
    ))

else:
    test_meta["pred_label_sw"] = np.nan
    test_meta["conf_sw"] = np.nan
    test_meta["n_windows"] = np.nan
    test_meta["votes_N"] = np.nan
    test_meta["votes_Neu"] = np.nan
    test_meta["votes_P"] = np.nan

# =========================================
# 13) Export Excel (test_all + errores simple + errores sliding)
# =========================================
cols_excel = [
    "id_intervencion", "file_name", "n_legislatura", "cuerpo", "fecha",
    "locutor", "encabezado", "intervencion", "n_palabras", "n_caracteres",
    "año", "sexo_locutor", "apellido_locutor_norm", "partido_imputado",
    "sentimiento", "text",
    "true_label",
    "pred_label", "conf_pred", "proba_N", "proba_Neu", "proba_P", "proba_vector",
    "pred_label_sw", "conf_sw", "n_windows", "votes_N", "votes_Neu", "votes_P",
]
cols_excel = [c for c in cols_excel if c in test_meta.columns]

df_errors_simple = test_meta[test_meta["true_label"] != test_meta["pred_label"]].copy()
df_errors_sliding = (
    test_meta[test_meta["true_label"] != test_meta["pred_label_sw"]].copy()
    if USE_SLIDING else pd.DataFrame()
)
df_corrige_sliding = (
    test_meta[
        (test_meta["true_label"] != test_meta["pred_label"]) &
        (test_meta["true_label"] == test_meta["pred_label_sw"])
    ].copy()
    if USE_SLIDING else pd.DataFrame()
)

out_path = f"analisis_{RUN_NAME}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    test_meta[cols_excel].to_excel(writer, index=False, sheet_name="test_all")
    df_errors_simple[cols_excel].to_excel(writer, index=False, sheet_name="errores_simple")
    if USE_SLIDING:
        df_errors_sliding[cols_excel].to_excel(writer, index=False, sheet_name="errores_sliding")
        df_corrige_sliding[cols_excel].to_excel(writer, index=False, sheet_name="corrige_sliding")

print("Excel generado:", out_path)
print(f"  test total:        {len(test_meta)}")
print(f"  errores simple:    {len(df_errors_simple)}")
if USE_SLIDING:
    print(f"  errores sliding:   {len(df_errors_sliding)}")
    print(f"  corrige sliding:   {len(df_corrige_sliding)}")

print("\n=== TEST SIMPLE ===")
print(classification_report(y_true, y_pred, target_names=label_list, zero_division=0))

if USE_SLIDING:
    print("\n=== TEST SLIDING ===")
    print(classification_report(y_true, y_pred_sw, target_names=label_list, zero_division=0))

print("\n" + "=" * 60)
print("RESUMEN FINAL — TEST")
print(f"Simple  | Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f} | "
      f"F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f} | "
      f"F1-weighted: {test_simple_metrics['test/f1_weighted_simple']:.4f}")

if USE_SLIDING:
    print(f"Sliding | Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f} | "
          f"F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f} | "
          f"F1-weighted: {test_sliding_metrics['test/f1_weighted_sliding']:.4f}")
print("=" * 60)

wandb.finish()

RUN: rouberta_slide_2050_vote ID: ng0shnlr USE_SLIDING: True AGG: vote
Train: 979 Val: 122 Test: 123


Map:   0%|          | 0/979 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


VALIDATION SIMPLE
  Accuracy: 0.6721
  F1-macro: 0.6642
              precision    recall  f1-score   support

           N       0.69      0.61      0.65        41
         Neu       0.68      0.80      0.73        45
           P       0.64      0.58      0.61        36

    accuracy                           0.67       122
   macro avg       0.67      0.66      0.66       122
weighted avg       0.67      0.67      0.67       122


VALIDATION SLIDING
  Accuracy: 0.7049
  F1-macro: 0.6907
              precision    recall  f1-score   support

           N       0.67      0.80      0.73        41
         Neu       0.69      0.80      0.74        45
           P       0.81      0.47      0.60        36

    accuracy                           0.70       122
   macro avg       0.73      0.69      0.69       122
weighted avg       0.72      0.70      0.70       122




TEST SIMPLE
  Accuracy: 0.8130
  F1-macro: 0.8118
              precision    recall  f1-score   support

           N       0.74      0.83      0.78        41
         Neu       0.85      0.85      0.85        46
           P       0.87      0.75      0.81        36

    accuracy                           0.81       123
   macro avg       0.82      0.81      0.81       123
weighted avg       0.82      0.81      0.81       123


TEST SLIDING
  Accuracy: 0.7724
  F1-macro: 0.7618
              precision    recall  f1-score   support

           N       0.68      0.88      0.77        41
         Neu       0.81      0.85      0.83        46
           P       0.91      0.56      0.69        36

    accuracy                           0.77       123
   macro avg       0.80      0.76      0.76       123
weighted avg       0.80      0.77      0.77       123

Excel generado: analisis_rouberta_slide_2050_vote.xlsx
  test total:        123
  errores simple:    23
  errores sliding:   28
  corri

data/n_test,▁
data/n_train,▁
data/n_val,▁
eval/accuracy,▁███
eval/accuracy_simple,▁
eval/accuracy_sliding,▁
eval/avg_windows_sliding,▁
eval/class_N_accuracy_ovr_simple,▁
eval/class_N_accuracy_ovr_sliding,▁
eval/class_N_f1_simple,▁
+88,...


## Slide + Mean

In [ ]:
# =========================================
# 0) W&B + Config (antes de todo)
# =========================================
import os
import random
import numpy as np
import pandas as pd
import torch
import wandb

WANDB_PROJECT = "Tesis ORT"
WANDB_ENTITY  = "fbenedetti97-universidad-ort-uruguay"

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["WANDB_ENTITY"] = WANDB_ENTITY

if wandb.run is not None:
    wandb.finish()

USE_SLIDING = True
AGG_METHOD  = "mean"   # "mean" / "vote"

# =========================================
# 1) Imports + Config
# =========================================
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

SEED       = 2050
MODEL_CKPT = "pln-udelar/rouberta-base-uy22-cased"
MODEL_TAG  = "rouberta"

RUN_NAME   = (
    f"{MODEL_TAG}_base_{SEED}"
    if not USE_SLIDING
    else f"{MODEL_TAG}_slide_{SEED}_{AGG_METHOD}"
)

MAX_LEN_TRAIN = 128
MAX_LEN_SW    = 128
STRIDE_SW     = 96

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================
# 2) Init W&B
# =========================================
run = wandb.init(
    project=WANDB_PROJECT,
    # entity=WANDB_ENTITY,
    name=RUN_NAME,
    config={
        "seed": int(SEED),
        "model_ckpt": MODEL_CKPT,
        "model_tag": MODEL_TAG,
        "run_type": "finetuned_single_run",
        "max_len_train": int(MAX_LEN_TRAIN),
        "use_sliding": USE_SLIDING,
        "max_len_sw": int(MAX_LEN_SW),
        "stride_sw": int(STRIDE_SW),
        "agg_method": AGG_METHOD,
        "learning_rate": float(2e-5),
        "train_batch_size": int(16),
        "eval_batch_size": int(32),
        "epochs": int(4),
        "weight_decay": float(0.01),
        "selection_metric": "test/f1_macro_sliding" if USE_SLIDING else "test/f1_macro_simple",
    },
    tags=["finetune", "single_run", "rouberta", "sliding" if USE_SLIDING else "simple"],
    reinit="finish_previous",
)

print("RUN:", wandb.run.name, "ID:", wandb.run.id, "USE_SLIDING:", USE_SLIDING, "AGG:", AGG_METHOD)

# =========================================
# 3) Utils
# =========================================
def softmax_stable(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

@torch.no_grad()
def predict_proba_windows(
    text: str,
    model,
    tokenizer,
    max_len: int = 128,
    stride: int = 64,
    device=None
):
    model.eval()
    if device is None:
        device = model.device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    if len(input_ids) > 0 and isinstance(input_ids[0], int):
        input_ids = [input_ids]
        attention_mask = [attention_mask]

    max_batch_len = max(len(x) for x in input_ids)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

    def pad(seq, pad_value):
        return seq + [pad_value] * (max_batch_len - len(seq))

    input_ids = torch.tensor(
        [pad(x, pad_id) for x in input_ids],
        dtype=torch.long,
        device=device
    )
    attention_mask = torch.tensor(
        [pad(x, 0) for x in attention_mask],
        dtype=torch.long,
        device=device
    )

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    return probs, len(probs)

@torch.no_grad()
def predict_text_truncated(text, model, tokenizer, id2label, max_len=128, device=None):
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        return_attention_mask=True,
        padding="max_length",
        return_tensors="pt",
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()[0]

    pred_id = int(np.argmax(probs))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs))
    n_win = 1
    votes_counts = None

    return pred_label, conf, probs, n_win, votes_counts

def aggregate_probs(probs_windows: np.ndarray, method: str = "mean"):
    if method == "mean":
        return probs_windows.mean(axis=0)
    if method == "max":
        return probs_windows.max(axis=0)
    if method == "vote":
        votes = np.argmax(probs_windows, axis=1)
        counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(float)
        return counts / counts.sum()
    raise ValueError("method debe ser 'mean', 'max' o 'vote'")

def predict_text_with_sliding(text, model, tokenizer, id2label,
                              max_len=128, stride=64, agg="mean"):
    """
    Retorna:
      pred_label (str),
      conf (float),
      probs_agg (np.array),
      n_win (int),
      votes_counts (np.array int) o None
    """
    probs_windows, n_win = predict_proba_windows(
        text=text,
        model=model,
        tokenizer=tokenizer,
        max_len=max_len,
        stride=stride
    )
    votes_counts = None

    if agg == "vote":
        votes = np.argmax(probs_windows, axis=1)
        votes_counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(int)
        max_count = votes_counts.max()
        tied = np.where(votes_counts == max_count)[0]

        if len(tied) == 1:
            pred_id = int(tied[0])
        else:
            mean_probs = probs_windows.mean(axis=0)
            pred_id = int(tied[np.argmax(mean_probs[tied])])

        probs_agg = votes_counts.astype(float) / votes_counts.sum()
        conf = float(probs_agg[pred_id])
        pred_label = id2label[pred_id]
        return pred_label, conf, probs_agg, n_win, votes_counts

    probs_agg = aggregate_probs(probs_windows, method=agg)
    pred_id = int(np.argmax(probs_agg))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs_agg))
    return pred_label, conf, probs_agg, n_win, votes_counts

# =========================================
# 4) Datos
# =========================================
df_lab = df_limpio[df_limpio["sentimiento"].notna()].copy()
df_lab["text"] = df_lab["intervencion"].astype(str)

valid_labels = ["N", "Neu", "P"]
df_lab = df_lab[df_lab["sentimiento"].isin(valid_labels)].copy()

label_list = ["N", "Neu", "P"]
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
df_lab["label"] = df_lab["sentimiento"].map(label2id)

# Split 80/10/10
train_df, temp_df = train_test_split(
    df_lab,
    test_size=0.20,
    random_state=SEED,
    stratify=df_lab["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

wandb.log({
    "data/n_train": len(train_df),
    "data/n_val": len(val_df),
    "data/n_test": len(test_df),
})

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# =========================================
# 5) Tokenize
# =========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize_train(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN_TRAIN)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

ds = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})
ds_tok = ds.map(tokenize_train, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================================
# 6) Model
# =========================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# =========================================
# 7) Metrics para Trainer
# =========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# =========================================
# 8) Trainer + Train
# =========================================
use_fp16 = torch.cuda.is_available()
OUTPUT_DIR = f"{MODEL_TAG}_finetuned_{wandb.run.id}"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    fp16=use_fp16,
    report_to="wandb",
    run_name=wandb.run.name,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

best_model = trainer.model

# =========================================
# 9) VALIDATION simple — referencia, no selección final
# =========================================
pred_val_simple = trainer.predict(ds_tok["validation"])
logits_val_simple = pred_val_simple.predictions
y_true_val_simple = pred_val_simple.label_ids
y_pred_val_simple = np.argmax(logits_val_simple, axis=-1)

eval_simple_metrics, eval_simple_report = build_metrics_dict(
    y_true=y_true_val_simple,
    y_pred=y_pred_val_simple,
    label_list=label_list,
    label2id=label2id,
    split="eval",
    variant="simple"
)

wandb.log(eval_simple_metrics)

print("\nVALIDATION SIMPLE")
print(f"  Accuracy: {eval_simple_metrics['eval/accuracy_simple']:.4f}")
print(f"  F1-macro: {eval_simple_metrics['eval/f1_macro_simple']:.4f}")
print(classification_report(
    y_true_val_simple,
    y_pred_val_simple,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 10) VALIDATION sliding — referencia, no selección final
# =========================================
if USE_SLIDING:
    y_true_val_sliding = val_df["label"].to_numpy()
    y_pred_val_sw = []
    nwin_val = []

    for txt in val_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_val_sw.append(label2id[lab])
        nwin_val.append(n_win)

    y_pred_val_sw = np.array(y_pred_val_sw)

    eval_sliding_metrics, eval_sliding_report = build_metrics_dict(
        y_true=y_true_val_sliding,
        y_pred=y_pred_val_sw,
        label_list=label_list,
        label2id=label2id,
        split="eval",
        variant="sliding"
    )

    eval_sliding_metrics.update({
        "eval/avg_windows_sliding": float(np.mean(nwin_val)),
        "eval/max_windows_sliding": float(np.max(nwin_val)),
    })

    wandb.log(eval_sliding_metrics)

    print("\nVALIDATION SLIDING")
    print(f"  Accuracy: {eval_sliding_metrics['eval/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {eval_sliding_metrics['eval/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true_val_sliding,
        y_pred_val_sw,
        target_names=label_list,
        zero_division=0
    ))

# =========================================
# 11) TEST simple — resultado final comparable
# =========================================
pred_test_simple = trainer.predict(ds_tok["test"])
logits_test_simple = pred_test_simple.predictions
y_true = pred_test_simple.label_ids
y_pred = np.argmax(logits_test_simple, axis=-1)
probs = softmax_stable(logits_test_simple, axis=1)

true_label_txt = np.array([id2label[i] for i in y_true])
pred_label_txt = np.array([id2label[i] for i in y_pred])

test_meta = test_df.reset_index(drop=True).copy()
test_meta["true_label"] = true_label_txt
test_meta["pred_label"] = pred_label_txt
test_meta["conf_pred"] = probs.max(axis=1)
test_meta["proba_N"] = probs[:, label2id["N"]]
test_meta["proba_Neu"] = probs[:, label2id["Neu"]]
test_meta["proba_P"] = probs[:, label2id["P"]]
test_meta["proba_vector"] = probs.tolist()

test_simple_metrics, test_simple_report = build_metrics_dict(
    y_true=y_true,
    y_pred=y_pred,
    label_list=label_list,
    label2id=label2id,
    split="test",
    variant="simple"
)

wandb.log(test_simple_metrics)

print("\nTEST SIMPLE")
print(f"  Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f}")
print(f"  F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f}")
print(classification_report(
    y_true,
    y_pred,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 12) TEST sliding — resultado final comparable
# =========================================
if USE_SLIDING:
    y_pred_sw = []
    conf_sw = []
    nwin_sw = []
    votes_N_list, votes_Neu_list, votes_P_list = [], [], []

    for txt in test_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_sw.append(label2id[lab])
        conf_sw.append(conf)
        nwin_sw.append(n_win)

        if votes_counts is None:
            votes_N_list.append(np.nan)
            votes_Neu_list.append(np.nan)
            votes_P_list.append(np.nan)
        else:
            votes_N_list.append(int(votes_counts[label2id["N"]]))
            votes_Neu_list.append(int(votes_counts[label2id["Neu"]]))
            votes_P_list.append(int(votes_counts[label2id["P"]]))

    y_pred_sw = np.array(y_pred_sw)

    assert len(test_df) == len(y_true) == len(y_pred_sw), "Mismatch en longitudes de test"

    test_meta["pred_label_sw"] = [id2label[i] for i in y_pred_sw]
    test_meta["conf_sw"] = conf_sw
    test_meta["n_windows"] = nwin_sw
    test_meta["votes_N"] = votes_N_list
    test_meta["votes_Neu"] = votes_Neu_list
    test_meta["votes_P"] = votes_P_list

    test_sliding_metrics, test_sliding_report = build_metrics_dict(
        y_true=y_true,
        y_pred=y_pred_sw,
        label_list=label_list,
        label2id=label2id,
        split="test",
        variant="sliding"
    )

    test_sliding_metrics.update({
        "test/avg_windows_sliding": float(np.mean(nwin_sw)),
        "test/max_windows_sliding": float(np.max(nwin_sw)),
    })

    wandb.log(test_sliding_metrics)

    print("\nTEST SLIDING")
    print(f"  Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true,
        y_pred_sw,
        target_names=label_list,
        zero_division=0
    ))

else:
    test_meta["pred_label_sw"] = np.nan
    test_meta["conf_sw"] = np.nan
    test_meta["n_windows"] = np.nan
    test_meta["votes_N"] = np.nan
    test_meta["votes_Neu"] = np.nan
    test_meta["votes_P"] = np.nan

# =========================================
# 13) Export Excel (test_all + errores simple + errores sliding)
# =========================================
cols_excel = [
    "id_intervencion", "file_name", "n_legislatura", "cuerpo", "fecha",
    "locutor", "encabezado", "intervencion", "n_palabras", "n_caracteres",
    "año", "sexo_locutor", "apellido_locutor_norm", "partido_imputado",
    "sentimiento", "text",
    "true_label",
    "pred_label", "conf_pred", "proba_N", "proba_Neu", "proba_P", "proba_vector",
    "pred_label_sw", "conf_sw", "n_windows", "votes_N", "votes_Neu", "votes_P",
]
cols_excel = [c for c in cols_excel if c in test_meta.columns]

df_errors_simple = test_meta[test_meta["true_label"] != test_meta["pred_label"]].copy()
df_errors_sliding = (
    test_meta[test_meta["true_label"] != test_meta["pred_label_sw"]].copy()
    if USE_SLIDING else pd.DataFrame()
)
df_corrige_sliding = (
    test_meta[
        (test_meta["true_label"] != test_meta["pred_label"]) &
        (test_meta["true_label"] == test_meta["pred_label_sw"])
    ].copy()
    if USE_SLIDING else pd.DataFrame()
)

out_path = f"analisis_{RUN_NAME}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    test_meta[cols_excel].to_excel(writer, index=False, sheet_name="test_all")
    df_errors_simple[cols_excel].to_excel(writer, index=False, sheet_name="errores_simple")
    if USE_SLIDING:
        df_errors_sliding[cols_excel].to_excel(writer, index=False, sheet_name="errores_sliding")
        df_corrige_sliding[cols_excel].to_excel(writer, index=False, sheet_name="corrige_sliding")

print("Excel generado:", out_path)
print(f"  test total:        {len(test_meta)}")
print(f"  errores simple:    {len(df_errors_simple)}")
if USE_SLIDING:
    print(f"  errores sliding:   {len(df_errors_sliding)}")
    print(f"  corrige sliding:   {len(df_corrige_sliding)}")

print("\n=== TEST SIMPLE ===")
print(classification_report(y_true, y_pred, target_names=label_list, zero_division=0))

if USE_SLIDING:
    print("\n=== TEST SLIDING ===")
    print(classification_report(y_true, y_pred_sw, target_names=label_list, zero_division=0))

print("\n" + "=" * 60)
print("RESUMEN FINAL — TEST")
print(f"Simple  | Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f} | "
      f"F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f} | "
      f"F1-weighted: {test_simple_metrics['test/f1_weighted_simple']:.4f}")

if USE_SLIDING:
    print(f"Sliding | Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f} | "
          f"F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f} | "
          f"F1-weighted: {test_sliding_metrics['test/f1_weighted_sliding']:.4f}")
print("=" * 60)

wandb.finish()

RUN: rouberta_slide_2050_mean ID: vmml3bxq USE_SLIDING: True AGG: mean
Train: 979 Val: 122 Test: 123


Map:   0%|          | 0/979 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


VALIDATION SIMPLE
  Accuracy: 0.6721
  F1-macro: 0.6642
              precision    recall  f1-score   support

           N       0.69      0.61      0.65        41
         Neu       0.68      0.80      0.73        45
           P       0.64      0.58      0.61        36

    accuracy                           0.67       122
   macro avg       0.67      0.66      0.66       122
weighted avg       0.67      0.67      0.67       122


VALIDATION SLIDING
  Accuracy: 0.7213
  F1-macro: 0.7051
              precision    recall  f1-score   support

           N       0.69      0.85      0.76        41
         Neu       0.72      0.80      0.76        45
           P       0.81      0.47      0.60        36

    accuracy                           0.72       122
   macro avg       0.74      0.71      0.71       122
weighted avg       0.74      0.72      0.71       122




TEST SIMPLE
  Accuracy: 0.8130
  F1-macro: 0.8118
              precision    recall  f1-score   support

           N       0.74      0.83      0.78        41
         Neu       0.85      0.85      0.85        46
           P       0.87      0.75      0.81        36

    accuracy                           0.81       123
   macro avg       0.82      0.81      0.81       123
weighted avg       0.82      0.81      0.81       123


TEST SLIDING
  Accuracy: 0.7886
  F1-macro: 0.7806
              precision    recall  f1-score   support

           N       0.69      0.90      0.78        41
         Neu       0.83      0.85      0.84        46
           P       0.95      0.58      0.72        36

    accuracy                           0.79       123
   macro avg       0.82      0.78      0.78       123
weighted avg       0.82      0.79      0.79       123

Excel generado: analisis_rouberta_slide_2050_mean.xlsx
  test total:        123
  errores simple:    23
  errores sliding:   26
  corri

data/n_test,▁
data/n_train,▁
data/n_val,▁
eval/accuracy,▁███
eval/accuracy_simple,▁
eval/accuracy_sliding,▁
eval/avg_windows_sliding,▁
eval/class_N_accuracy_ovr_simple,▁
eval/class_N_accuracy_ovr_sliding,▁
eval/class_N_f1_simple,▁
+88,...


# Random Search

In [ ]:
# =========================================
# 0) DEBUG CUDA
# =========================================
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# =========================================
# 1) Imports + Config general
# =========================================
import gc
import random
import shutil
import traceback
import numpy as np
import pandas as pd
import wandb
import torch

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
)

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

# =========================================
# 2) Config global
# =========================================
WANDB_PROJECT = "Tesis ORT"
WANDB_ENTITY  = "fbenedetti97-universidad-ort-uruguay"  # opcional

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["WANDB_ENTITY"] = WANDB_ENTITY  # descomentá si aplica

if wandb.run is not None:
    wandb.finish()

BASE_SEED  = 1899
MODEL_CKPT = "pln-udelar/rouberta-base-uy22-cased"
MODEL_TAG  = "rouberta"
DATA_PATH  = Path("/content/df_final_para_modelo.jsonl")  # opcional / informativo

USE_SLIDING = True

# --- Random search ---
N_RANDOM_RUNS = 3
RUN_TEST_EACH_EXPERIMENT = True  # False: test solo con el mejor config al final

# --- Multi-seed final ---
FINAL_SEEDS = [1899, 1405, 2050]

DELETE_OUTPUT_DIR = True

# Métrica oficial para elegir la mejor config en FASE 1
SELECTION_METRIC = "eval_f1_macro_sliding" if USE_SLIDING else "eval_f1_macro_simple"

# =========================================
# 3) Utils
# =========================================
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def softmax_stable(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def assert_labels_ok(df, df_name="df"):
    assert df["label"].notna().all(), f"{df_name}: hay labels NaN"
    uniq = set(df["label"].unique())
    assert uniq <= {0, 1, 2}, f"{df_name}: labels fuera de rango -> {uniq}"

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def normalize_config_types(cfg: dict):
    return {
        "learning_rate":    float(cfg["learning_rate"]),
        "epochs":           int(cfg["epochs"]),
        "train_batch_size": int(cfg["train_batch_size"]),
        "eval_batch_size":  int(cfg["eval_batch_size"]),
        "weight_decay":     float(cfg["weight_decay"]),
        "warmup_ratio":     float(cfg["warmup_ratio"]),
        "max_len_train":    int(cfg["max_len_train"]),
        "max_len_sw":       int(cfg["max_len_sw"]),
        "stride_sw":        int(cfg["stride_sw"]),
        "agg_method":       str(cfg["agg_method"]),
    }

@torch.no_grad()
def predict_proba_windows(text: str, model, tokenizer, max_len: int = 128, stride: int = 64, device=None):
    """
    Sliding windows usando overflow del tokenizer.
    stride = cantidad de tokens solapados entre ventanas.
    """
    model.eval()
    if device is None:
        device = model.device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    if len(input_ids) > 0 and isinstance(input_ids[0], int):
        input_ids = [input_ids]
        attention_mask = [attention_mask]

    max_batch_len = max(len(x) for x in input_ids)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

    def pad(seq, pad_value):
        return seq + [pad_value] * (max_batch_len - len(seq))

    input_ids = torch.tensor(
        [pad(x, pad_id) for x in input_ids],
        dtype=torch.long,
        device=device
    )
    attention_mask = torch.tensor(
        [pad(x, 0) for x in attention_mask],
        dtype=torch.long,
        device=device
    )

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()

    return probs, len(probs)

def aggregate_probs(probs_windows: np.ndarray, method: str = "mean"):
    if method == "mean":
        return probs_windows.mean(axis=0)
    if method == "vote":
        votes = np.argmax(probs_windows, axis=1)
        counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(float)
        return counts / counts.sum()
    raise ValueError("method debe ser 'mean' o 'vote'")

def predict_text_with_sliding(text, model, tokenizer, id2label, max_len=128, stride=64, agg="mean"):
    """
    Retorna:
      pred_label   (str)
      conf         (float)
      probs_agg    (np.array)
      n_win        (int)
      votes_counts (np.array int) o None
    """
    probs_windows, n_win = predict_proba_windows(
        text=text,
        model=model,
        tokenizer=tokenizer,
        max_len=max_len,
        stride=stride
    )

    votes_counts = None

    if agg == "vote":
        votes = np.argmax(probs_windows, axis=1)
        votes_counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(int)

        max_count = votes_counts.max()
        tied = np.where(votes_counts == max_count)[0]

        if len(tied) == 1:
            pred_id = int(tied[0])
        else:
            mean_probs = probs_windows.mean(axis=0)
            pred_id = int(tied[np.argmax(mean_probs[tied])])

        probs_agg = votes_counts.astype(float) / votes_counts.sum()
        conf = float(probs_agg[pred_id])
        pred_label = id2label[pred_id]
        return pred_label, conf, probs_agg, n_win, votes_counts

    probs_agg = aggregate_probs(probs_windows, method=agg)
    pred_id = int(np.argmax(probs_agg))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs_agg))
    return pred_label, conf, probs_agg, n_win, votes_counts

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# =========================================
# 4) Load data (una sola vez)
# =========================================
df_lab = df_limpio[df_limpio["sentimiento"].notna()].copy()
df_lab["text"] = df_lab["intervencion"].astype(str)

valid_labels = ["N", "Neu", "P"]
df_lab = df_lab[df_lab["sentimiento"].isin(valid_labels)].copy()

label_list = ["N", "Neu", "P"]
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
df_lab["label"] = df_lab["sentimiento"].map(label2id)

print("Labels únicas en df_lab:", sorted(df_lab["label"].unique().tolist()))
print("Conteo labels:")
print(df_lab["label"].value_counts(dropna=False).sort_index())

assert_labels_ok(df_lab, "df_lab")

train_df, temp_df = train_test_split(
    df_lab,
    test_size=0.20,
    random_state=BASE_SEED,
    stratify=df_lab["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=BASE_SEED,
    stratify=temp_df["label"]
)

assert_labels_ok(train_df, "train_df")
assert_labels_ok(val_df, "val_df")
assert_labels_ok(test_df, "test_df")

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# =========================================
# 5) Espacio de búsqueda
# =========================================
SEARCH_SPACE = {
    "learning_rate":    [1e-5, 1.5e-5, 2e-5, 3e-5],
    "epochs":           [3, 4, 5],
    "train_batch_size": [8, 16],
    "eval_batch_size":  [32],
    "weight_decay":     [0.0, 0.01, 0.05],
    "warmup_ratio":     [0.0, 0.1],
    "max_len_train":    [128, 256],
    "max_len_sw":       [128, 256],
    "agg_method":       ["mean", "vote"],
}

STRIDE_BY_MAXLEN = {
    128: [32, 64, 96],
    256: [64, 128, 192],
}

def sample_unique_configs(n_runs: int, seed: int = 1899):
    rng = random.Random(seed)
    seen = set()
    configs = []

    while len(configs) < n_runs:
        max_len_sw = rng.choice(SEARCH_SPACE["max_len_sw"])

        cfg = {
            "learning_rate":    rng.choice(SEARCH_SPACE["learning_rate"]),
            "epochs":           rng.choice(SEARCH_SPACE["epochs"]),
            "train_batch_size": rng.choice(SEARCH_SPACE["train_batch_size"]),
            "eval_batch_size":  rng.choice(SEARCH_SPACE["eval_batch_size"]),
            "weight_decay":     rng.choice(SEARCH_SPACE["weight_decay"]),
            "warmup_ratio":     rng.choice(SEARCH_SPACE["warmup_ratio"]),
            "max_len_train":    rng.choice(SEARCH_SPACE["max_len_train"]),
            "max_len_sw":       max_len_sw,
            "stride_sw":        rng.choice(STRIDE_BY_MAXLEN[max_len_sw]),
            "agg_method":       rng.choice(SEARCH_SPACE["agg_method"]),
        }

        key = tuple(sorted(cfg.items()))
        if key not in seen:
            seen.add(key)
            configs.append(cfg)

    return configs

random_configs = sample_unique_configs(N_RANDOM_RUNS, seed=BASE_SEED)

print("\nConfiguraciones sorteadas:")
for i, cfg in enumerate(random_configs, 1):
    print(f"  {i}. {cfg}")

# =========================================
# 6) Función principal por experimento
# =========================================
def run_one_experiment(cfg: dict, exp_idx: int, seed: int, run_prefix: str = "rs"):
    """
    Entrena una sola corrida con la config y seed dadas.

    run_prefix:
      - "rs"    -> random search (fase 1)
      - "final" -> multi-seed final (fase 2)

    En fase 1 se usa VALIDATION para selección de configs.
    En fase 2 se corre TEST para evaluación final.
    """
    cfg = normalize_config_types(cfg)
    set_all_seeds(seed)

    run_name = (
        f"{MODEL_TAG}_{run_prefix}_{exp_idx:02d}"
        f"_seed{seed}"
        f"_agg{cfg['agg_method']}"
        f"_lr{cfg['learning_rate']}"
        f"_mlt{cfg['max_len_train']}"
        f"_mls{cfg['max_len_sw']}"
        f"_st{cfg['stride_sw']}"
    )

    if run_prefix == "rs":
        run_tags = ["random_search", "validation_only", MODEL_TAG, "sliding" if USE_SLIDING else "simple"]
        selection_metric_value = SELECTION_METRIC
    else:
        run_tags = ["final_multiseed", "test", MODEL_TAG, "sliding" if USE_SLIDING else "simple"]
        selection_metric_value = "test/f1_macro_sliding" if USE_SLIDING else "test/f1_macro_simple"

    run = wandb.init(
        project=WANDB_PROJECT,
        # entity=WANDB_ENTITY,
        name=run_name,
        config={
            "seed": int(seed),
            "model_ckpt": MODEL_CKPT,
            "model_tag": MODEL_TAG,
            "use_sliding": USE_SLIDING,
            "run_prefix": run_prefix,
            "selection_metric": selection_metric_value,
            **cfg,
        },
        tags=run_tags,
        reinit="finish_previous",
    )

    print("\n" + "=" * 90)
    print(f"RUN: {run.name} | ID: {run.id} | prefix: {run_prefix}")
    print(cfg)

    wandb.log({
        "data/n_train": len(train_df),
        "data/n_val": len(val_df),
        "data/n_test": len(test_df),
    })

    output_dir = f"{MODEL_TAG}_{run_prefix}_{run.id}"

    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

        def tokenize_fn(batch):
            return tokenizer(batch["text"], truncation=True, max_length=cfg["max_len_train"])

        train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
        val_ds   = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
        test_ds  = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

        ds = DatasetDict({
            "train": train_ds,
            "validation": val_ds,
            "test": test_ds
        })
        ds_tok = ds.map(tokenize_fn, batched=True)

        data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

        assert set(ds_tok["train"]["label"]) <= {0, 1, 2}
        assert set(ds_tok["validation"]["label"]) <= {0, 1, 2}
        assert set(ds_tok["test"]["label"]) <= {0, 1, 2}

        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_CKPT,
            num_labels=len(label_list),
            id2label=id2label,
            label2id=label2id,
            ignore_mismatched_sizes=True,
        )

        assert model.config.num_labels == 3

        use_fp16 = torch.cuda.is_available()

        args = TrainingArguments(
            output_dir=output_dir,
            learning_rate=cfg["learning_rate"],
            per_device_train_batch_size=cfg["train_batch_size"],
            per_device_eval_batch_size=cfg["eval_batch_size"],
            num_train_epochs=cfg["epochs"],
            weight_decay=cfg["weight_decay"],
            warmup_ratio=cfg["warmup_ratio"],
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="f1_macro",
            greater_is_better=True,
            seed=seed,
            logging_steps=50,
            fp16=use_fp16,
            report_to="wandb",
            run_name=run.name,
        )

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=ds_tok["train"],
            eval_dataset=ds_tok["validation"],
            processing_class=tokenizer,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
        )

        trainer.train()

        best_model = trainer.model

        # =========================================
        # Métricas auxiliares del trainer (informativas)
        # =========================================
        eval_logs = [x for x in trainer.state.log_history if "eval_f1_macro" in x]
        if eval_logs:
            best_eval_trainer = max(eval_logs, key=lambda x: x["eval_f1_macro"])
        else:
            best_eval_trainer = {}

        result_row = {
            "status": "ok",
            "run_name": run.name,
            "run_id": run.id,
            "seed": seed,
            "run_prefix": run_prefix,
            **cfg,
            "trainer_best_eval_loss": best_eval_trainer.get("eval_loss"),
            "trainer_best_eval_accuracy": best_eval_trainer.get("eval_accuracy"),
            "trainer_best_eval_f1_macro": best_eval_trainer.get("eval_f1_macro"),
            "trainer_best_eval_f1_weighted": best_eval_trainer.get("eval_f1_weighted"),
            "trainer_best_eval_epoch": best_eval_trainer.get("epoch"),
        }

        # =========================================
        # VALIDATION simple — oficial para comparación interna
        # =========================================
        pred_val_simple = trainer.predict(ds_tok["validation"])
        logits_val_simple = pred_val_simple.predictions
        y_true_val_simple = pred_val_simple.label_ids
        y_pred_val_simple = np.argmax(logits_val_simple, axis=-1)

        eval_simple_metrics, eval_simple_report = build_metrics_dict(
            y_true=y_true_val_simple,
            y_pred=y_pred_val_simple,
            label_list=label_list,
            label2id=label2id,
            split="eval",
            variant="simple"
        )

        wandb.log(eval_simple_metrics)

        result_row.update({
            "eval_accuracy_simple": eval_simple_metrics["eval/accuracy_simple"],
            "eval_f1_macro_simple": eval_simple_metrics["eval/f1_macro_simple"],
            "eval_f1_weighted_simple": eval_simple_metrics["eval/f1_weighted_simple"],
        })

        for lbl in label_list:
            result_row[f"eval_class_{lbl}_precision_simple"] = eval_simple_metrics[f"eval/class_{lbl}_precision_simple"]
            result_row[f"eval_class_{lbl}_recall_simple"] = eval_simple_metrics[f"eval/class_{lbl}_recall_simple"]
            result_row[f"eval_class_{lbl}_f1_simple"] = eval_simple_metrics[f"eval/class_{lbl}_f1_simple"]
            result_row[f"eval_class_{lbl}_support_simple"] = eval_simple_metrics[f"eval/class_{lbl}_support_simple"]
            result_row[f"eval_class_{lbl}_accuracy_ovr_simple"] = eval_simple_metrics[f"eval/class_{lbl}_accuracy_ovr_simple"]

        # =========================================
        # VALIDATION sliding — métrica principal si USE_SLIDING=True
        # =========================================
        if USE_SLIDING:
            y_true_val_sliding = val_df["label"].to_numpy()
            y_pred_val_sw_list = []
            nwin_val_list = []

            for txt in val_df["text"].tolist():
                lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
                    txt,
                    best_model,
                    tokenizer,
                    id2label,
                    max_len=cfg["max_len_sw"],
                    stride=cfg["stride_sw"],
                    agg=cfg["agg_method"],
                )
                y_pred_val_sw_list.append(label2id[lab])
                nwin_val_list.append(n_win)

            y_pred_val_sw = np.array(y_pred_val_sw_list)

            eval_sliding_metrics, eval_sliding_report = build_metrics_dict(
                y_true=y_true_val_sliding,
                y_pred=y_pred_val_sw,
                label_list=label_list,
                label2id=label2id,
                split="eval",
                variant="sliding"
            )

            eval_sliding_metrics.update({
                "eval/avg_windows_sliding": float(np.mean(nwin_val_list)),
                "eval/max_windows_sliding": float(np.max(nwin_val_list)),
            })

            wandb.log(eval_sliding_metrics)

            result_row.update({
                "eval_accuracy_sliding": eval_sliding_metrics["eval/accuracy_sliding"],
                "eval_f1_macro_sliding": eval_sliding_metrics["eval/f1_macro_sliding"],
                "eval_f1_weighted_sliding": eval_sliding_metrics["eval/f1_weighted_sliding"],
                "eval_avg_windows_sliding": eval_sliding_metrics["eval/avg_windows_sliding"],
                "eval_max_windows_sliding": eval_sliding_metrics["eval/max_windows_sliding"],
            })

            for lbl in label_list:
                result_row[f"eval_class_{lbl}_precision_sliding"] = eval_sliding_metrics[f"eval/class_{lbl}_precision_sliding"]
                result_row[f"eval_class_{lbl}_recall_sliding"] = eval_sliding_metrics[f"eval/class_{lbl}_recall_sliding"]
                result_row[f"eval_class_{lbl}_f1_sliding"] = eval_sliding_metrics[f"eval/class_{lbl}_f1_sliding"]
                result_row[f"eval_class_{lbl}_support_sliding"] = eval_sliding_metrics[f"eval/class_{lbl}_support_sliding"]
                result_row[f"eval_class_{lbl}_accuracy_ovr_sliding"] = eval_sliding_metrics[f"eval/class_{lbl}_accuracy_ovr_sliding"]

        # =========================================
        # TEST solo en fase final (o si se fuerza explícitamente)
        # =========================================
        run_test = (run_prefix == "final") or RUN_TEST_EACH_EXPERIMENT

        if run_test:
            # -----------------------------------------
            # TEST simple
            # -----------------------------------------
            pred_test_simple = trainer.predict(ds_tok["test"])
            logits_test_simple = pred_test_simple.predictions
            y_true = pred_test_simple.label_ids
            y_pred = np.argmax(logits_test_simple, axis=-1)

            test_simple_metrics, test_simple_report = build_metrics_dict(
                y_true=y_true,
                y_pred=y_pred,
                label_list=label_list,
                label2id=label2id,
                split="test",
                variant="simple"
            )

            wandb.log(test_simple_metrics)

            result_row.update({
                "test_accuracy_simple": test_simple_metrics["test/accuracy_simple"],
                "test_f1_macro_simple": test_simple_metrics["test/f1_macro_simple"],
                "test_f1_weighted_simple": test_simple_metrics["test/f1_weighted_simple"],
            })

            for lbl in label_list:
                result_row[f"test_class_{lbl}_precision_simple"] = test_simple_metrics[f"test/class_{lbl}_precision_simple"]
                result_row[f"test_class_{lbl}_recall_simple"] = test_simple_metrics[f"test/class_{lbl}_recall_simple"]
                result_row[f"test_class_{lbl}_f1_simple"] = test_simple_metrics[f"test/class_{lbl}_f1_simple"]
                result_row[f"test_class_{lbl}_support_simple"] = test_simple_metrics[f"test/class_{lbl}_support_simple"]
                result_row[f"test_class_{lbl}_accuracy_ovr_simple"] = test_simple_metrics[f"test/class_{lbl}_accuracy_ovr_simple"]

            # -----------------------------------------
            # TEST sliding
            # -----------------------------------------
            if USE_SLIDING:
                y_pred_sw_list = []
                conf_sw_list = []
                nwin_sw_list = []

                for txt in test_df["text"].tolist():
                    lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
                        txt,
                        best_model,
                        tokenizer,
                        id2label,
                        max_len=cfg["max_len_sw"],
                        stride=cfg["stride_sw"],
                        agg=cfg["agg_method"],
                    )
                    y_pred_sw_list.append(label2id[lab])
                    conf_sw_list.append(conf)
                    nwin_sw_list.append(n_win)

                y_pred_sw = np.array(y_pred_sw_list)

                test_sliding_metrics, test_sliding_report = build_metrics_dict(
                    y_true=y_true,
                    y_pred=y_pred_sw,
                    label_list=label_list,
                    label2id=label2id,
                    split="test",
                    variant="sliding"
                )

                test_sliding_metrics.update({
                    "test/avg_windows_sliding": float(np.mean(nwin_sw_list)),
                    "test/max_windows_sliding": float(np.max(nwin_sw_list)),
                })

                wandb.log(test_sliding_metrics)

                result_row.update({
                    "test_accuracy_sliding": test_sliding_metrics["test/accuracy_sliding"],
                    "test_f1_macro_sliding": test_sliding_metrics["test/f1_macro_sliding"],
                    "test_f1_weighted_sliding": test_sliding_metrics["test/f1_weighted_sliding"],
                    "test_avg_windows_sliding": test_sliding_metrics["test/avg_windows_sliding"],
                    "test_max_windows_sliding": test_sliding_metrics["test/max_windows_sliding"],
                })

                for lbl in label_list:
                    result_row[f"test_class_{lbl}_precision_sliding"] = test_sliding_metrics[f"test/class_{lbl}_precision_sliding"]
                    result_row[f"test_class_{lbl}_recall_sliding"] = test_sliding_metrics[f"test/class_{lbl}_recall_sliding"]
                    result_row[f"test_class_{lbl}_f1_sliding"] = test_sliding_metrics[f"test/class_{lbl}_f1_sliding"]
                    result_row[f"test_class_{lbl}_support_sliding"] = test_sliding_metrics[f"test/class_{lbl}_support_sliding"]
                    result_row[f"test_class_{lbl}_accuracy_ovr_sliding"] = test_sliding_metrics[f"test/class_{lbl}_accuracy_ovr_sliding"]

        wandb.finish()

        del trainer, model, best_model, ds_tok, ds, train_ds, val_ds, test_ds
        cleanup_cuda()

        if DELETE_OUTPUT_DIR and os.path.exists(output_dir):
            shutil.rmtree(output_dir, ignore_errors=True)

        return result_row

    except Exception as e:
        err_trace = traceback.format_exc()
        print("\n[ERROR EN RUN]")
        print(type(e).__name__, str(e))
        print(err_trace)

        try:
            wandb.log({
                "error": 1,
                "error_type": type(e).__name__,
                "error_msg": str(e),
            })
            wandb.finish()
        except:
            pass

        cleanup_cuda()

        if DELETE_OUTPUT_DIR and os.path.exists(output_dir):
            shutil.rmtree(output_dir, ignore_errors=True)

        return {
            "status": "error",
            "run_name": run.name if wandb.run is not None else run_name,
            "run_id": run.id if "run" in locals() else None,
            "seed": seed,
            "run_prefix": run_prefix,
            **cfg,
            "error_type": type(e).__name__,
            "error_msg": str(e),
        }

# =========================================
# 7) Random search
# =========================================
print("\n" + "=" * 90)
print(f"FASE 1 — RANDOM SEARCH ({N_RANDOM_RUNS} runs, seed fija={BASE_SEED})")
print(f"Métrica de selección: {SELECTION_METRIC}")
print("=" * 90)

rs_results = []

for i, cfg in enumerate(random_configs, start=1):
    row = run_one_experiment(cfg, exp_idx=i, seed=BASE_SEED, run_prefix="rs")
    rs_results.append(row)

rs_df = pd.DataFrame(rs_results)

sort_col = SELECTION_METRIC

if sort_col in rs_df.columns:
    rs_df = rs_df.sort_values(sort_col, ascending=False, na_position="last").reset_index(drop=True)

rs_df.to_csv(f"{MODEL_TAG}_random_search_resultados.csv", index=False)
print(f"\nCSV guardado: {MODEL_TAG}_random_search_resultados.csv")

cols_show = [
    "run_name",
    sort_col,
    "eval_f1_macro_simple",
    "eval_f1_macro_sliding",
    "learning_rate",
    "epochs",
    "train_batch_size",
    "weight_decay",
    "warmup_ratio",
    "max_len_train",
    "max_len_sw",
    "stride_sw",
    "agg_method",
]
print(rs_df[[c for c in cols_show if c in rs_df.columns]])

# =========================================
# 8) Seleccionar mejor config
# =========================================
rs_ok = rs_df[rs_df["status"] == "ok"].copy()

if len(rs_ok) == 0:
    raise RuntimeError("No hubo corridas exitosas en el random search.")

best_row = rs_ok.sort_values(sort_col, ascending=False).iloc[0]

cfg_keys = list(SEARCH_SPACE.keys()) + ["stride_sw"]
best_cfg_raw = {k: best_row[k] for k in cfg_keys if k in best_row}
best_cfg = normalize_config_types(best_cfg_raw)

print("\n" + "=" * 90)
print("MEJOR CONFIGURACIÓN (por validation):")
for k, v in best_cfg.items():
    print(f"  {k}: {v}")
print(f"  {sort_col}: {best_row[sort_col]:.4f}")

print("\nTipos de best_cfg:")
for k, v in best_cfg.items():
    print(f"  {k}: {type(v)}")

# =========================================
# 9) Multi-seed final con la mejor config
# =========================================
print("\n" + "=" * 90)
print(f"FASE 2 — MULTI-SEED FINAL ({len(FINAL_SEEDS)} seeds: {FINAL_SEEDS})")
print("Misma config, distinta seed -> estimación de varianza del fine-tuning")
print("Estas corridas SÍ corren test set.")
print("=" * 90)

final_results = []

for i, seed in enumerate(FINAL_SEEDS, start=1):
    row = run_one_experiment(best_cfg, exp_idx=i, seed=seed, run_prefix="final")
    final_results.append(row)

final_df = pd.DataFrame(final_results)
final_df.to_csv(f"{MODEL_TAG}_final_multiseed_resultados.csv", index=False)
print(f"\nCSV guardado: {MODEL_TAG}_final_multiseed_resultados.csv")

# =========================================
# 10) Resumen estadístico final
# =========================================
final_ok = final_df[final_df["status"] == "ok"].copy()

print("\n" + "=" * 90)
print("RESUMEN MULTI-SEED (TEST SET)")

report_cols = [
    "test_f1_macro_simple",
    "test_f1_weighted_simple",
    "test_accuracy_simple",
    "test_f1_macro_sliding",
    "test_f1_weighted_sliding",
    "test_accuracy_sliding",
]

for col in report_cols:
    if col in final_ok.columns and final_ok[col].notna().any():
        vals = final_ok[col].dropna().values
        print(
            f"  {col}: mean={np.mean(vals):.4f}  std={np.std(vals):.4f}  "
            f"min={np.min(vals):.4f}  max={np.max(vals):.4f}"
        )

print("\nDone.")

Labels únicas en df_lab: [0, 1, 2]
Conteo labels:
label
0    410
1    452
2    362
Name: count, dtype: int64
Train: 979 | Val: 122 | Test: 123

Configuraciones sorteadas:
  1. {'learning_rate': 1e-05, 'epochs': 3, 'train_batch_size': 16, 'eval_batch_size': 32, 'weight_decay': 0.0, 'warmup_ratio': 0.0, 'max_len_train': 128, 'max_len_sw': 128, 'stride_sw': 64, 'agg_method': 'vote'}
  2. {'learning_rate': 1e-05, 'epochs': 5, 'train_batch_size': 16, 'eval_batch_size': 32, 'weight_decay': 0.0, 'warmup_ratio': 0.1, 'max_len_train': 128, 'max_len_sw': 128, 'stride_sw': 96, 'agg_method': 'vote'}
  3. {'learning_rate': 1e-05, 'epochs': 3, 'train_batch_size': 8, 'eval_batch_size': 32, 'weight_decay': 0.01, 'warmup_ratio': 0.1, 'max_len_train': 128, 'max_len_sw': 128, 'stride_sw': 64, 'agg_method': 'vote'}

FASE 1 — RANDOM SEARCH (3 runs, seed fija=1899)
Métrica de selección: eval_f1_macro_sliding



RUN: rouberta_rs_01_seed1899_aggvote_lr1e-05_mlt128_mls128_st64 | ID: cxb2o58d | prefix: rs
{'learning_rate': 1e-05, 'epochs': 3, 'train_batch_size': 16, 'eval_batch_size': 32, 'weight_decay': 0.0, 'warmup_ratio': 0.0, 'max_len_train': 128, 'max_len_sw': 128, 'stride_sw': 64, 'agg_method': 'vote'}


Map:   0%|          | 0/979 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

data/n_test,▁
data/n_train,▁
data/n_val,▁
eval/accuracy,▁▆▆▇█
eval/accuracy_simple,▁
eval/accuracy_sliding,▁
eval/avg_windows_sliding,▁
eval/class_N_accuracy_ovr_simple,▁
eval/class_N_accuracy_ovr_sliding,▁
eval/class_N_f1_simple,▁
+88,...



CSV guardado: rouberta_final_multiseed_resultados.csv

RESUMEN MULTI-SEED (TEST SET)
  test_f1_macro_simple: mean=0.7231  std=0.0246  min=0.7004  max=0.7574
  test_f1_weighted_simple: mean=0.7306  std=0.0236  min=0.7081  max=0.7633
  test_accuracy_simple: mean=0.7317  std=0.0239  min=0.7073  max=0.7642
  test_f1_macro_sliding: mean=0.6337  std=0.0227  min=0.6015  max=0.6511
  test_f1_weighted_sliding: mean=0.6473  std=0.0191  min=0.6202  max=0.6617
  test_accuracy_sliding: mean=0.6775  std=0.0077  min=0.6667  max=0.6829

Done.
